In [3]:
# A simple turn-based sci fi game



# game setup for new game

# hero selection
import json
import os
import datetime
import uuid
import copy

# ══════════════════════════════════════════════
#  FILE PATHS
# ══════════════════════════════════════════════
SAVE_DIR  = "saves"
SAVE_FILE = os.path.join(SAVE_DIR, "heroes.json")
os.makedirs(SAVE_DIR, exist_ok=True)


# ══════════════════════════════════════════════
#  LOOKUP TABLES  (number → label)
# ══════════════════════════════════════════════
HERO_TYPES = {
    "1": "Sorcerer",
    "2": "Thug",
    "3": "Slayer",
    "4": "Sneak",
    "5": "Genius",
}

GIFTS = {
    "1": "Speech",
    "2": "Speed",
    "3": "Physical Strength",
    "4": "Sight",
    "5": "Love",
}

MAGIC_SYSTEMS = {
    "1": "Fire",
    "2": "Water",
    "3": "Earth",
    "4": "Air",
    "5": "Lightning",
    "6": "Darkness",
    "7": "Nano",
}

HERO_HOMES = {
    "1": "Forest",
    "2": "Desert",
    "3": "Mountain",
    "4": "Sea",
    "5": "Sky",
}

# Starting stat bonuses per hero type
TYPE_BONUSES = {
    "Sorcerer": {"magic_power": 3,  "intelligence": 2, "stealth": 0,  "strength": 0,  "charisma": 0},
    "Thug":     {"magic_power": 0,  "intelligence": 0, "stealth": 0,  "strength": 3,  "charisma": 1},
    "Slayer":   {"magic_power": 1,  "intelligence": 0, "stealth": 1,  "strength": 2,  "charisma": 0},
    "Sneak":    {"magic_power": 0,  "intelligence": 1, "stealth": 3,  "strength": 0,  "charisma": 1},
    "Genius":   {"magic_power": 1,  "intelligence": 3, "stealth": 0,  "strength": 0,  "charisma": 1},
}

# Starting stat bonuses per gift
GIFT_BONUSES = {
    "Speech":           {"charisma": 3},
    "Speed":            {"agility": 3},
    "Physical Strength":{"strength": 3},
    "Sight":            {"perception": 3},
    "Love":             {"willpower": 3},
}

# Home environment flavour (affects starting item + lore)
HOME_LORE = {
    "Forest":   "raised among ancient trees and hidden paths",
    "Desert":   "hardened by endless heat and scarce water",
    "Mountain": "forged in cold stone and thin air",
    "Sea":      "born on open water, navigator by instinct",
    "Sky":      "a child of the clouds, unbounded and restless",
}


# ══════════════════════════════════════════════
#  TIMESTAMP HELPER
# ══════════════════════════════════════════════
def _timestamp():
    return datetime.datetime.now().isoformat(timespec="seconds")


# ══════════════════════════════════════════════
#  DEFAULT PLAYER BUILDER
# ══════════════════════════════════════════════
def build_hero(name, hero_type, gift, magic, home):
    """Construct a full hero record from hero_selection() choices."""

    # Base attributes
    base_attrs = {
        "strength":    5,
        "intelligence":5,
        "stealth":     5,
        "charisma":    5,
        "agility":     5,
        "willpower":   5,
        "perception":  5,
        "magic_power": 5,
    }

    # Apply type bonus
    for stat, bonus in TYPE_BONUSES.get(hero_type, {}).items():
        base_attrs[stat] = base_attrs.get(stat, 5) + bonus

    # Apply gift bonus
    for stat, bonus in GIFT_BONUSES.get(gift, {}).items():
        base_attrs[stat] = base_attrs.get(stat, 5) + bonus

    # Starting item based on home
    home_items = {
        "Forest":   {"name": "Carved Bow",       "type": "weapon",    "value": 40,
                     "stats": {"agility": 1},
                     "description": "Bent from a living branch. Still grows."},
        "Desert":   {"name": "Sand Cloak",        "type": "armor",     "value": 35,
                     "stats": {"stealth": 1},
                     "description": "Blends with dunes. Keeps the heat out."},
        "Mountain": {"name": "Iron Pick",         "type": "weapon",    "value": 38,
                     "stats": {"strength": 1},
                     "description": "Used for climbing first, fighting second."},
        "Sea":      {"name": "Navigator's Chart", "type": "key_item",  "value": 50,
                     "stats": {"perception": 1},
                     "description": "Shows coasts no map-maker ever drew."},
        "Sky":      {"name": "Wind Charm",        "type": "relic",     "value": 45,
                     "stats": {"agility": 1, "magic_power": 1},
                     "description": "Hums when a storm is three days out."},
    }
    start_item = home_items.get(home, {})
    if start_item:
        start_item["id"]       = str(uuid.uuid4())
        start_item["acquired"] = _timestamp()

    return {
        # ── Identity ──
        "id":          str(uuid.uuid4()),
        "name":        name,
        "hero_type":   hero_type,
        "gift":        gift,
        "magic":       magic,
        "home":        home,
        "home_lore":   HOME_LORE.get(home, ""),
        "created":     _timestamp(),
        "last_played": _timestamp(),

        # ── Progression ──
        "level":       1,
        "xp":          0,
        "xp_to_next":  100,

        # ── Vitals ──
        "hp":          100,
        "hp_max":      100,
        "mana":        60,
        "mana_max":    60,
        "stamina":     50,
        "stamina_max": 50,

        # ── Currency ──
        "gold":        100,
        "rare_essence":0,      # drops from powerful enemies

        # ── Attributes ──
        "attributes":  base_attrs,

        # ── Magic ──
        "magic_system": {
            "element":    magic,
            "spells":     [f"{magic} Bolt"],   # starting spell
            "mastery":    1,                    # 1-10
        },

        # ── Inventory ──
        "inventory": [start_item] if start_item else [],

        # ── Equipped ──
        "equipped": {
            "weapon":   None,
            "armor":    None,
            "relic":    None,
            "accessory":None,
        },

        # ── Story flags ──
        "flags": {
            "intro_complete":    False,
            "act":               1,
            "quests_complete":   [],
            "quests_active":     [],
            "choices":           [],
            "world_events":      [],
        },

        # ── Reputation ──
        "reputation": {
            "kingdom":    0,
            "thieves":    0,
            "mages":      0,
            "wilds":      0,
            "underworld": 0,
        },

        # ── Lifetime stats ──
        "stats": {
            "quests_complete":  0,
            "quests_failed":    0,
            "enemies_defeated": 0,
            "spells_cast":      0,
            "times_died":       0,
            "gold_earned":      0,
            "play_time_seconds":0,
        },

        # ── Save slots ──
        "save_slots": {},
    }


# ══════════════════════════════════════════════
#  DATABASE HELPERS
# ══════════════════════════════════════════════
def _load_db():
    if not os.path.exists(SAVE_FILE):
        return {}
    try:
        with open(SAVE_FILE, "r") as f:
            content = f.read().strip()
            return json.loads(content) if content else {}
    except (json.JSONDecodeError, OSError) as e:
        print(f"[SAVE] Warning — could not read save file: {e}")
        return {}

def _write_db(db):
    try:
        with open(SAVE_FILE, "w") as f:
            json.dump(db, f, indent=2)
    except OSError as e:
        print(f"[SAVE] Error — could not write save file: {e}")


# ══════════════════════════════════════════════
#  PLAYER CRUD
# ══════════════════════════════════════════════
def save_hero(hero):
    db = _load_db()
    hero["last_played"] = _timestamp()
    db[hero["name"]] = hero
    _write_db(db)
    print(f"[SAVE] '{hero['name']}' saved.")

def load_hero(name):
    db = _load_db()
    hero = db.get(name)
    if hero is None:
        print(f"[SAVE] Hero '{name}' not found.")
        return None
    hero["last_played"] = _timestamp()
    _write_db(db)
    print(f"[SAVE] Hero '{name}' loaded.")
    return hero

def delete_hero(name):
    db = _load_db()
    if name not in db:
        print(f"[SAVE] Hero '{name}' not found.")
        return False
    del db[name]
    _write_db(db)
    print(f"[SAVE] Hero '{name}' deleted.")
    return True

def hero_exists(name):
    return name in _load_db()

def list_heroes():
    db = _load_db()
    heroes = []
    for h in db.values():
        heroes.append({
            "name":        h["name"],
            "hero_type":   h["hero_type"],
            "gift":        h["gift"],
            "magic":       h["magic"],
            "home":        h["home"],
            "level":       h["level"],
            "act":         h["flags"]["act"],
            "last_played": h["last_played"],
        })
    return sorted(heroes, key=lambda x: x["last_played"], reverse=True)


# ══════════════════════════════════════════════
#  QUICK-SAVE SLOTS
# ══════════════════════════════════════════════
def quick_save(hero, slot="slot_1"):
    snapshot = copy.deepcopy(hero)
    snapshot.pop("save_slots", None)
    hero["save_slots"][slot] = {
        "timestamp": _timestamp(),
        "data":      snapshot,
    }
    save_hero(hero)
    print(f"[SAVE] Quick-save written to '{slot}'.")

def quick_load(hero, slot="slot_1"):
    slots = hero.get("save_slots", {})
    if slot not in slots:
        print(f"[SAVE] Slot '{slot}' not found.")
        return None
    restored = copy.deepcopy(slots[slot]["data"])
    restored["save_slots"] = hero["save_slots"]
    print(f"[SAVE] Loaded from slot '{slot}' (saved {slots[slot]['timestamp']}).")
    return restored

def list_save_slots(hero):
    slots = hero.get("save_slots", {})
    if not slots:
        print("  No quick-save slots found.")
        return
    for slot, info in slots.items():
        d = info["data"]
        print(f"  [{slot}]  Level {d['level']}  |  "
              f"Act {d['flags']['act']}  |  "
              f"Saved: {info['timestamp']}")


# ══════════════════════════════════════════════
#  PROGRESSION
# ══════════════════════════════════════════════
def add_xp(hero, amount):
    hero["xp"] += amount
    leveled = False
    while hero["xp"] >= hero["xp_to_next"]:
        hero["xp"]         -= hero["xp_to_next"]
        hero["level"]      += 1
        hero["xp_to_next"]  = int(hero["xp_to_next"] * 1.35)
        hero["hp_max"]     += 12
        hero["hp"]          = hero["hp_max"]
        hero["mana_max"]   += 8
        hero["stamina_max"]+= 5
        hero["magic_system"]["mastery"] = min(10, hero["magic_system"]["mastery"] + 1)
        leveled = True
        print(f"  ★ LEVEL UP → {hero['level']}  "
              f"(HP {hero['hp_max']}  Mana {hero['mana_max']})")
    if not leveled:
        print(f"  +{amount} XP  ({hero['xp']}/{hero['xp_to_next']})")
    return leveled

def modify_attribute(hero, attr, delta):
    if attr not in hero["attributes"]:
        print(f"[RPG] Unknown attribute: {attr}")
        return
    old = hero["attributes"][attr]
    hero["attributes"][attr] = max(1, min(20, old + delta))
    print(f"  {attr.capitalize()}  {old} → {hero['attributes'][attr]}")

def modify_reputation(hero, faction, delta):
    if faction not in hero["reputation"]:
        print(f"[RPG] Unknown faction: {faction}")
        return
    old = hero["reputation"][faction]
    hero["reputation"][faction] = max(-100, min(100, old + delta))
    new = hero["reputation"][faction]
    print(f"  {faction.capitalize()} reputation  {old:+d} → {new:+d}  ({_rep_label(new)})")

def _rep_label(rep):
    if rep >=  80: return "LEGENDARY"
    if rep >=  40: return "Trusted"
    if rep >=  10: return "Neutral+"
    if rep >= -10: return "Neutral"
    if rep >= -40: return "Suspicious"
    if rep >= -80: return "Hostile"
    return "HUNTED"

def complete_quest(hero, quest_id, xp=0, gold=0):
    if quest_id in hero["flags"]["quests_complete"]:
        print(f"[RPG] Quest '{quest_id}' already completed.")
        return
    hero["flags"]["quests_complete"].append(quest_id)
    hero["stats"]["quests_complete"] += 1
    if gold:
        hero["gold"] += gold
        hero["stats"]["gold_earned"] += gold
        print(f"  +{gold} gold")
    if xp:
        add_xp(hero, xp)
    print(f"  Quest '{quest_id}' complete.")

def learn_spell(hero, spell_name):
    spells = hero["magic_system"]["spells"]
    if spell_name not in spells:
        spells.append(spell_name)
        print(f"  Learned spell: {spell_name}")
    else:
        print(f"  Already knows: {spell_name}")

def set_flag(hero, key, value):
    hero["flags"][key] = value
    print(f"  [FLAG] {key} = {value}")

def log_choice(hero, description):
    hero["flags"]["choices"].append({"time": _timestamp(), "event": description})
    print(f"  [STORY] {description}")


# ══════════════════════════════════════════════
#  INVENTORY
# ══════════════════════════════════════════════
def make_item(name, item_type, value=0, stats=None, description=""):
    return {
        "id":          str(uuid.uuid4()),
        "name":        name,
        "type":        item_type,
        "value":       value,
        "stats":       stats or {},
        "description": description,
        "acquired":    _timestamp(),
    }

def add_item(hero, item):
    hero["inventory"].append(item)
    print(f"  [INV] Added: {item['name']}")

def remove_item(hero, item_name):
    for i, item in enumerate(hero["inventory"]):
        if item["name"].lower() == item_name.lower():
            hero["inventory"].pop(i)
            print(f"  [INV] Removed: {item_name}")
            return True
    print(f"  [INV] '{item_name}' not found.")
    return False

def equip_item(hero, item_name):
    for item in hero["inventory"]:
        if item["name"].lower() == item_name.lower():
            slot = item["type"]
            if slot not in hero["equipped"]:
                print(f"  [INV] Can't equip type '{slot}'.")
                return
            hero["equipped"][slot] = item
            print(f"  [INV] Equipped '{item['name']}' → slot '{slot}'.")
            return
    print(f"  [INV] '{item_name}' not in inventory.")

def show_inventory(hero):
    print(f"\n── Inventory ({len(hero['inventory'])} items) ──")
    if not hero["inventory"]:
        print("  (empty)")
    for item in hero["inventory"]:
        stats = "  ".join(f"{k}+{v}" for k,v in item["stats"].items())
        print(f"  • {item['name']:<26} [{item['type']:<10}]  "
              f"${item['value']:<5}  {stats}")
    print("── Equipped ──")
    for slot, item in hero["equipped"].items():
        print(f"  {slot:<12} {item['name'] if item else '—'}")


# ══════════════════════════════════════════════
#  DISPLAY
# ══════════════════════════════════════════════
def show_hero(hero):
    a  = hero["attributes"]
    ms = hero["magic_system"]
    rp = hero["reputation"]
    st = hero["stats"]
    fl = hero["flags"]

    print("\n" + "═"*56)
    print(f"  {hero['name'].upper()}  ·  {hero['hero_type'].upper()}")
    print(f"  Gift: {hero['gift']}  ·  Magic: {ms['element']}  "
          f"·  From: {hero['home']}")
    print(f"  \"{hero['home_lore']}\"")
    print("─"*56)
    print(f"  Level {hero['level']}  ·  XP {hero['xp']}/{hero['xp_to_next']}  "
          f"·  Act {fl['act']}")
    print(f"  HP      {hero['hp']}/{hero['hp_max']}   "
          f"Mana    {hero['mana']}/{hero['mana_max']}   "
          f"Stamina {hero['stamina']}/{hero['stamina_max']}")
    print(f"  Gold    {hero['gold']}   Rare Essence  {hero['rare_essence']}")
    print("─"*56)
    print("  ATTRIBUTES")
    for attr, val in a.items():
        bar = "█"*val + "░"*(20-val)
        print(f"    {attr:<18} {bar}  {val}")
    print("─"*56)
    print(f"  MAGIC — {ms['element'].upper()}  "
          f"(Mastery {ms['mastery']}/10)")
    print(f"  Spells known: {', '.join(ms['spells'])}")
    print("─"*56)
    print("  REPUTATION")
    for fac, rep in rp.items():
        label = _rep_label(rep)
        print(f"    {fac:<14} {rep:>+4}  {label}")
    print("─"*56)
    print("  LIFETIME STATS")
    print(f"    Quests  {st['quests_complete']}✓  {st['quests_failed']}✗  "
          f"│  Enemies  {st['enemies_defeated']}  "
          f"│  Deaths  {st['times_died']}")
    print(f"    Spells cast  {st['spells_cast']}  "
          f"│  Gold earned  {st['gold_earned']}")
    print("─"*56)
    print(f"  Last played: {hero['last_played']}")
    print("═"*56 + "\n")

print(show_hero)
# ══════════════════════════════════════════════
#  HERO SELECTION  (your original code, extended)
# ══════════════════════════════════════════════
def _pick(prompt, options):
    """Show a numbered menu and return the resolved string value."""
    print(f"\n{prompt}")
    for k, v in options.items():
        print(f"  [{k}] {v}")
    while True:
        choice = input(">> ").strip()
        if choice in options:
            return options[choice]
        # Also accept typing the name directly
        for v in options.values():
            if choice.lower() == v.lower():
                return v
        print(f"  Please enter a number 1–{len(options)}.")

def hero_selection():
    hero_name  = input("\nHero name: ").strip()
    hero_type  = _pick("Select your hero type:",  HERO_TYPES)
    hero_gift  = _pick("Select your hero's gift:", GIFTS)
    magic      = _pick("Select your magic system:", MAGIC_SYSTEMS)
    hero_home  = _pick("Select your hero's home:", HERO_HOMES)
    return hero_name, hero_type, hero_gift, magic, hero_home


# ══════════════════════════════════════════════
#  MAIN ENTRY POINT
# ══════════════════════════════════════════════
if __name__ == "__main__":

    print("╔══════════════════════════════════════════╗")
    print("║       WELCOME TO THE HERO'S JOURNEY      ║")
    print("╚══════════════════════════════════════════╝")
    print("Please select your hero.\n")

    # ── Check for existing heroes first ──
    existing = list_heroes()
    hero = None

    if existing:
        print("── Existing heroes found ──")
        for i, h in enumerate(existing, 1):
            print(f"  [{i}] {h['name']:<16} {h['hero_type']:<10} "
                  f"Lv{h['level']}  Act {h['act']}  "
                  f"Last: {h['last_played']}")
        print(f"  [N] Create a new hero")
        print(f"  [D] Delete a hero")

        choice = input("\n>> ").strip().upper()

        if choice == "N":
            pass  # fall through to creation below

        elif choice == "D":
            name = input("Hero name to delete: ").strip()
            delete_hero(name)
            # re-list and continue
            existing = list_heroes()

        elif choice.isdigit() and 1 <= int(choice) <= len(existing):
            hero = load_hero(existing[int(choice)-1]["name"])

        else:
            # Maybe they typed a name
            hero = load_hero(choice) if not choice.isdigit() else None

    # ── Create new hero if none loaded ──
    if hero is None:
        hero_name, hero_type, hero_gift, magic, hero_home = hero_selection()

        if hero_exists(hero_name):
            print(f"\n  A hero named '{hero_name}' already exists.")
            overwrite = input("  Overwrite? (yes/no): ").strip().lower()
            if overwrite != "yes":
                print("  Keeping existing hero. Loading instead.")
                hero = load_hero(hero_name)
            else:
                hero = build_hero(hero_name, hero_type, hero_gift, magic, hero_home)
                save_hero(hero)
        else:
            hero = build_hero(hero_name, hero_type, hero_gift, magic, hero_home)
            save_hero(hero)

        # ── Your original confirmation line ──
        print(f"\nYour hero is {hero['name']}, a {hero['hero_type']} "
              f"with a gift for {hero['gift']} "
              f"and a magic system of {hero['magic']['element'] if isinstance(hero['magic'], dict) else hero['magic']} "
              f"from {hero['home']}.")

    # ── Show full profile ──
    show_hero(hero)

    # ── Equip starting weapon if one exists ──
    for item in hero["inventory"]:
        if item["type"] in ("weapon", "relic"):
            equip_item(hero, item["name"])
            break

    show_inventory(hero)

    # ── Demo: quick-save ──
    quick_save(hero, "slot_1")
    list_save_slots(hero)

    print("\nYou are ready to begin the epic journey!")
    print("─"*56)
    print("(Your hero is saved and will be here when you return.)")
    print("─"*56)

    # Final persist
    save_hero(hero)

# Logging user data


#introduction to the journey

import random
import pygame
import math

pygame.init()
pygame.mixer.init(frequency=44100, size=-16, channels=1, buffer=512)
W, H = 800, 450
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("2157")
clock = pygame.time.Clock()

# ── Palette ──
SKY_TOP  = (8,   4,  20)
SKY_BOT  = (25, 10,  45)
PINK     = (220, 40, 120)
CYAN     = (0,  200, 220)
PURPLE   = (120, 40, 200)
YELLOW   = (220, 180,  0)
DIM_CYAN = (0,   80,  90)
DIM_PINK = (80,  15,  45)
GRID_COL = (0,   60,  80)

# ── Crash sound (synthesised — no file needed) ──
def make_crash_sound():
    rate     = 44100
    duration = 2.2          # seconds
    n        = int(rate * duration)
    buf      = bytearray(n)

    for i in range(n):
        t   = i / rate
        env = max(0.0, 1.0 - t / duration) ** 0.4   # decay envelope

        # white-noise burst
        noise = (random.randint(0, 255) - 128) * env

        # low-freq thud at the start
        thud_freq = 55
        thud      = math.sin(2 * math.pi * thud_freq * t) * 127 * max(0, 1 - t * 4)

        # high screech that drops in pitch
        screech_freq = max(40, 1800 - t * 1400)
        screech      = math.sin(2 * math.pi * screech_freq * t) * 60 * env * min(1, t * 8)

        sample = int(noise * 0.55 + thud + screech)
        buf[i] = max(0, min(255, sample + 128))

    sound = pygame.mixer.Sound(buffer=bytes(buf))
    return sound

crash_sound = make_crash_sound()

# ── Scroll text ──
RAW_TEXT = (
    "THE WORLD — 2157 A.D.\n"
    "\n"
    "The year is 2157. Earth is unrecognizable\n"
    "yet strangely familiar.\n"
    "\n"
    "Three generations have passed since the\n"
    "Singularity Accord — the moment humanity's\n"
    "greatest minds, governments, and spiritual\n"
    "leaders convened to answer one question\n"
    "that could no longer be avoided:\n"
    "\n"
    "What do we do now that we can do anything?\n"
    "\n"
    "The answer fractured civilization into\n"
    "three distinct paths, known collectively\n"
    "as The Pitchfork — a term coined\n"
    "sarcastically by a late-night comedian\n"
    "in 2089 that somehow stuck, eventually\n"
    "becoming the official name in global\n"
    "census records.\n"
    "\n"
    "The three tines of the Pitchfork have\n"
    "since calcified into cultures, economies,\n"
    "ideologies, and wars.\n"
    "\n"
    "The player exists at the bleeding edge\n"
    "of all three.\n"
    "\n\n\n\n\n\n"          # padding so last line clears the screen
)

pygame.font.init()
FONT_TITLE  = pygame.font.SysFont("consolas", 15, bold=True)
FONT_BODY   = pygame.font.SysFont("consolas", 13)
FONT_HUD    = pygame.font.SysFont("consolas", 11)
FONT_HERO   = pygame.font.SysFont("consolas", 38, bold=True)
FONT_HERO_S = pygame.font.SysFont("consolas", 20, bold=True)

def make_text_surfaces(raw):
    surfaces = []
    for line in raw.split("\n"):
        stripped = line.strip()
        if stripped == "":
            surfaces.append(None)
            continue
        if stripped.startswith("THE WORLD"):
            surf = FONT_TITLE.render(line, True, CYAN)
        elif stripped.startswith("What do we"):
            surf = FONT_BODY.render(line, True, YELLOW)
        else:
            surf = FONT_BODY.render(line, True, (160, 200, 220))
        surfaces.append(surf)
    return surfaces

text_surfaces  = make_text_surfaces(RAW_TEXT)
LINE_H         = 22
SCROLL_SPEED   = 0.55
text_y         = float(H)
TEXT_X         = 30
TEXT_W         = W - 60
PANEL_ALPHA    = 170

# How far text_y must scroll before all lines have left the top
total_text_h   = len(text_surfaces) * LINE_H
# text is "done" when text_y + total_text_h < 0  →  text_y < -total_text_h
text_done      = False

# ── Stars ──
random.seed(42)
stars = [
    (random.randint(0,W), random.randint(0,H//2),
     random.random(), random.choice([CYAN,PINK,(180,180,220)]))
    for _ in range(120)
]

# ── Moon ──
moon = {"x":620,"y":80,"r":38}

# ── Buildings ──
def make_buildings():
    blds   = []
    ground = int(H*0.68)
    specs_back  = [(30,160),(90,210),(160,180),(230,230),(300,170),
                   (370,250),(440,190),(510,220),(580,160),(650,200),(710,175),(760,140)]
    specs_mid   = [(0,110),(60,130),(130,100),(200,120),(280,140),
                   (350,110),(420,130),(490,120),(560,100),(630,130),(700,115),(755,95)]
    specs_front = [(0,55),(70,65),(150,50),(230,70),(310,58),
                   (390,62),(460,55),(535,68),(610,52),(675,60),(735,55),(775,48)]
    for layer,(specs,wrange,wc) in enumerate([
        (specs_back, (44,68), PURPLE),
        (specs_mid,  (52,72), CYAN),
        (specs_front,(55,78), YELLOW),
    ]):
        cols = [(18,8,40),(12,6,28),(6,3,16)][layer]
        for x,h in specs:
            w = random.randint(*wrange)
            blds.append({
                "x":x,"y":ground-h,"w":w,"h":h,
                "color":cols,"layer":layer,"win_color":wc,
                "windows":[(random.randint(4,w-10),random.randint(6,h-10))
                           for _ in range(random.randint(2,12))
                           if random.random()>0.3]
            })
    blds.sort(key=lambda b:b["layer"])
    return blds, ground

buildings, GROUND_Y = make_buildings()
win_states = {i:[random.random()>0.15 for _ in b["windows"]]
              for i,b in enumerate(buildings)}

# ── Rain ──
drops = [{"x":random.uniform(0,W),"y":random.uniform(0,H),
           "speed":random.uniform(4,9),"length":random.randint(6,16),
           "color":random.choice([DIM_CYAN,DIM_PINK,(40,30,70)])}
          for _ in range(140)]

# ── Flyers ──
flyers = [{"x":random.uniform(0,W),"y":random.uniform(H*0.1,H*0.55),
            "speed":random.uniform(0.6,2.2)*random.choice([-1,1]),
            "color":random.choice([CYAN,YELLOW,PINK,(180,180,255)]),
            "size":random.randint(2,4),"blink":random.randint(20,60)}
           for _ in range(8)]

# ── Helpers ──
def draw_sky():
    for y in range(H):
        t = y/H
        r = int(SKY_TOP[0]+(SKY_BOT[0]-SKY_TOP[0])*t)
        g = int(SKY_TOP[1]+(SKY_BOT[1]-SKY_TOP[1])*t)
        b = int(SKY_TOP[2]+(SKY_BOT[2]-SKY_TOP[2])*t)
        pygame.draw.line(screen,(r,g,b),(0,y),(W,y))

def draw_stars(tick):
    for sx,sy,phase,sc in stars:
        bright = 0.5+0.5*math.sin(tick*0.05+phase*8)
        pygame.draw.circle(screen,tuple(int(c*bright) for c in sc),(sx,sy),1)

def draw_moon():
    for gr in range(22,0,-4):
        gs = pygame.Surface((gr*2,gr*2),pygame.SRCALPHA)
        pygame.draw.circle(gs,PINK+(int(40*gr/22),),(gr,gr),gr)
        screen.blit(gs,(moon["x"]-gr,moon["y"]-gr))
    pygame.draw.circle(screen,(35,12,80),(moon["x"],moon["y"]),moon["r"])
    pygame.draw.circle(screen,(55,18,110),(moon["x"],moon["y"]),moon["r"],2)
    for cx,cy,cr in [(608,72,7),(632,88,5),(618,95,4)]:
        pygame.draw.circle(screen,(28,9,65),(cx,cy),cr)

def draw_horizon(ground_y):
    for i in range(30):
        t=i/30; a=int(60*(1-t))
        col=(int(PINK[0]*(1-t)+PURPLE[0]*t),
             int(PINK[1]*(1-t)+PURPLE[1]*t),
             int(PINK[2]*(1-t)+PURPLE[2]*t))
        s=pygame.Surface((W,1),pygame.SRCALPHA); s.fill(col+(a,))
        screen.blit(s,(0,ground_y-i))

def draw_buildings(tick):
    for i,b in enumerate(buildings):
        pygame.draw.rect(screen,b["color"],(b["x"],b["y"],b["w"],b["h"]))
        edge=CYAN if b["layer"]==0 else(PINK if b["layer"]==1 else YELLOW)
        pygame.draw.line(screen,edge,(b["x"],b["y"]),(b["x"],b["y"]+b["h"]),1)
        for j,(wx,wy) in enumerate(b["windows"]):
            on=win_states[i][j]
            if tick%random.randint(90,300)==(i+j)%60:
                win_states[i][j]=not on; on=win_states[i][j]
            pygame.draw.rect(screen,b["win_color"] if on else(8,4,18),
                             (b["x"]+wx,b["y"]+wy,5,4))
        if b["layer"] in(0,1) and b["w"]>55:
            ax=b["x"]+b["w"]//2
            pygame.draw.line(screen,(80,80,100),(ax,b["y"]),(ax,b["y"]-14),1)
            pygame.draw.circle(screen,PINK if(tick//25)%2==0 else(30,10,30),(ax,b["y"]-14),2)

def draw_ground(tick):
    pygame.draw.rect(screen,(4,1,12),(0,GROUND_Y,W,H-GROUND_Y))
    vp=W//2
    for gx in range(0,W+1,55):
        pygame.draw.line(screen,GRID_COL,(vp,GROUND_Y),(gx,H),1)
    spacing=32; offset=(tick*2)%spacing
    for gy in range(0,H-GROUND_Y+spacing,spacing):
        t=gy/(H-GROUND_Y+spacing); br=int(80*t)
        col=(0,min(br,80),min(br+20,100))
        y=GROUND_Y+gy-offset
        if GROUND_Y<=y<=H: pygame.draw.line(screen,col,(0,y),(W,y),1)

def draw_rain():
    rs=pygame.Surface((W,H),pygame.SRCALPHA)
    for d in drops:
        d["y"]+=d["speed"]; d["x"]+=d["speed"]*0.12
        if d["y"]>H: d["y"]=random.uniform(-10,0); d["x"]=random.uniform(0,W)
        pygame.draw.line(rs,d["color"]+(55,),
                         (int(d["x"]),int(d["y"])),
                         (int(d["x"]+d["length"]*0.12),int(d["y"]+d["length"])),1)
    screen.blit(rs,(0,0))

def draw_flyers(tick):
    for f in flyers:
        f["x"]+=f["speed"]
        if f["x"]>W+20: f["x"]=-20
        if f["x"]<-20:  f["x"]=W+20
        if(tick//f["blink"])%2==0:
            pygame.draw.circle(screen,f["color"],(int(f["x"]),int(f["y"])),f["size"])

def draw_scanlines():
    for sy in range(0,H,4):
        pygame.draw.line(screen,(0,0,0),(0,sy),(W,sy),1)

def draw_scroll_text(text_y):
    panel=pygame.Surface((TEXT_W,H),pygame.SRCALPHA)
    panel.fill((4,1,14,PANEL_ALPHA))
    screen.blit(panel,(TEXT_X,0))
    for idx,surf in enumerate(text_surfaces):
        ly=int(text_y+idx*LINE_H)
        if -LINE_H<ly<H and surf is not None:
            fade=255
            if ly>H-60: fade=int(255*(H-ly)/60)
            if ly<60:   fade=int(255*ly/60)
            fade=max(0,min(255,fade))
            glow=surf.copy(); glow.set_alpha(max(0,fade-160))
            screen.blit(glow,(TEXT_X+10,ly+1))
            lx=TEXT_X+(TEXT_W-surf.get_width())//2
            surf.set_alpha(fade)
            screen.blit(surf,(lx,ly))

# ════════════════════════════════════════
# STATE MACHINE
# ════════════════════════════════════════
# "scroll"  → normal scrolling skyline
# "hero"    → big "THE HERO'S JOURNEY BEGINS" text
# "pixelate"→ white pixel static builds up
# "crash"   → full white flash then freeze
# "done"    → quit

state        = "scroll"
hero_alpha   = 0          # fades in
hero_timer   = 0          # frames spent in hero state
HERO_HOLD    = 210        # frames (~3.5 s) to show hero text

pixel_timer  = 0
PIXEL_PHASES = 120        # frames for pixelation to build
pixel_surf   = None       # offscreen for pixelated frame

crash_timer  = 0
CRASH_HOLD   = 90
crash_played = False

tick = 0
running = True

while running:
    clock.tick(60)
    tick += 1

    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False
        if e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE:
            running = False

    # ════════════════════════════════════
    # DRAW SKYLINE (always, as backdrop)
    # ════════════════════════════════════
    draw_sky()
    draw_stars(tick)
    draw_moon()
    draw_horizon(GROUND_Y)
    draw_buildings(tick)
    draw_ground(tick)
    draw_rain()
    draw_flyers(tick)
    draw_scanlines()

    # ════════════════════════════════════
    # STATE: scroll
    # ════════════════════════════════════
    if state == "scroll":
        text_y -= SCROLL_SPEED
        draw_scroll_text(text_y)

        # HUD
        hud = FONT_HUD.render("NEON DISTRICT  //  2157", True, (0,120,140))
        hud.set_alpha(100)
        screen.blit(hud,(8,8))

        # Check if all text has scrolled off the top
        if text_y + total_text_h < 0:
            text_done = True
            state     = "hero"
            hero_alpha = 0

    # ════════════════════════════════════
    # STATE: hero  — big title fades in
    # ════════════════════════════════════
    elif state == "hero":
        hero_timer += 1

        # Fade in over first 60 frames, hold, done
        if hero_timer < 60:
            hero_alpha = int(255 * hero_timer / 60)
        else:
            hero_alpha = 255

        # Dark vignette so text pops
        veil = pygame.Surface((W, H), pygame.SRCALPHA)
        veil.fill((0, 0, 0, 160))
        screen.blit(veil, (0, 0))

        # Pulsing glow behind text
        pulse = 0.7 + 0.3 * math.sin(tick * 0.08)

        line1 = "THE HERO'S"
        line2 = "JOURNEY BEGINS"

        for font, text, y_off, col in [
            (FONT_HERO,   line1, -52, CYAN),
            (FONT_HERO,   line2,   8, PINK),
            (FONT_HERO_S, "── PRESS ANY KEY ──", 78, (100,160,180)),
        ]:
            rendered = font.render(text, True, col)
            # glow pass
            glow_col = tuple(int(c * pulse * 0.6) for c in col)
            glow     = font.render(text, True, glow_col)
            cx = W//2 - rendered.get_width()//2
            cy = H//2 + y_off
            for ox,oy in [(-3,0),(3,0),(0,-3),(0,3)]:
                g=glow.copy(); g.set_alpha(int(80*pulse))
                screen.blit(g,(cx+ox,cy+oy))
            rendered.set_alpha(hero_alpha)
            screen.blit(rendered,(cx,cy))

        # Advance after hold, OR on any keypress
        keys = pygame.key.get_pressed()
        key_pressed = any(keys)

        if hero_timer >= HERO_HOLD or (hero_timer > 60 and key_pressed):
            state       = "pixelate"
            pixel_timer = 0
            # Snapshot current frame for pixelation
            pixel_surf  = screen.copy()

    # ════════════════════════════════════
    # STATE: pixelate — blocky static
    # ════════════════════════════════════
    elif state == "pixelate":
        pixel_timer += 1
        progress = pixel_timer / PIXEL_PHASES   # 0 → 1

        if not crash_played and progress > 0.3:
            crash_sound.play()
            crash_played = True

        # Draw the snapshot then blast random white/colour squares over it
        screen.blit(pixel_surf, (0, 0))

        # Block size grows as chaos increases
        block = max(2, int(4 + progress * 28))
        num_blocks = int(progress * (W * H) // (block * block) * 1.4)

        noise_surf = pygame.Surface((W, H), pygame.SRCALPHA)
        for _ in range(num_blocks):
            bx = random.randint(0, W - block)
            by = random.randint(0, H - block)
            # Mostly white, occasional colour
            if random.random() < 0.75:
                col = (255, 255, 255, random.randint(120, 255))
            else:
                col = random.choice([
                    CYAN+(random.randint(80,200),),
                    PINK+(random.randint(80,200),),
                    YELLOW+(random.randint(80,200),),
                    (255,255,255,255),
                ])
            noise_surf.fill(col, (bx, by, block, block))

        screen.blit(noise_surf, (0, 0))

        # Horizontal glitch bars
        if random.random() < 0.5 + progress * 0.5:
            for _ in range(int(progress * 12)):
                gy  = random.randint(0, H)
                gh  = random.randint(2, 8)
                gsh = int(progress * 40)
                glitch = pygame.Surface((W, gh), pygame.SRCALPHA)
                glitch.blit(screen, (-random.randint(0,gsh), -gy))
                col = random.choice([CYAN,PINK,YELLOW,(255,255,255)])+(160,)
                pygame.draw.rect(glitch, col, (0,0,W,gh), 1)
                screen.blit(glitch, (0, gy))

        # Screen-shake offset
        shake = int(progress * 9)
        if shake > 0:
            tmp = screen.copy()
            ox = random.randint(-shake, shake)
            oy = random.randint(-shake, shake)
            screen.fill((0,0,0))
            screen.blit(tmp, (ox, oy))

        if pixel_timer >= PIXEL_PHASES:
            state       = "crash"
            crash_timer = 0

    # ════════════════════════════════════
    # STATE: crash — full white, hold, quit
    # ════════════════════════════════════
    elif state == "crash":
        crash_timer += 1

        # Blinding white flash
        screen.fill((255, 255, 255))

        # After a beat, add a tiny "signal lost" message
        if crash_timer > 20:
            sig = FONT_HUD.render("// SIGNAL LOST //", True, (180,0,0))
            screen.blit(sig, (W//2 - sig.get_width()//2, H//2))

        if crash_timer >= CRASH_HOLD:
            state = "done"

    # ════════════════════════════════════
    elif state == "done":
        running = False

    pygame.display.flip()

# Brief pause so the white screen is visible before window closes
pygame.time.wait(400)
pygame.quit()

print("SURVIVE AT ALL COSTS!")

import pygame
import math
import random

pygame.init()
W, H = 900, 500
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("// STREET LEVEL — 2157 //")
clock = pygame.time.Clock()

# ── Palette ──
SKY_TOP    = (4,   2,  10)
SKY_BOT    = (14,  5,  28)
PINK       = (200, 30, 100)
CYAN       = (0,  180, 200)
YELLOW     = (200, 160,  0)
DIM_PINK   = (60,  10,  35)
DIM_CYAN   = (0,   50,  65)
GROUND_COL = (8,   4,  14)
PUDDLE_COL = (12,  6,  22)
SHADOW     = (2,   1,   6)

random.seed(7)

# ══════════════════════════════════════════
#  SCENE GEOMETRY
# ══════════════════════════════════════════
GROUND_Y  = int(H * 0.62)   # where street begins
HORIZON_Y = int(H * 0.42)   # vanishing horizon
VP_X      = W // 2           # vanishing point x

# ── Perspective helpers ──
def persp_x(world_x, depth):
    """Map a world x offset + depth to screen x (converges to VP_X)."""
    return int(VP_X + world_x / max(depth, 0.01))

def persp_y(world_y, depth):
    return int(HORIZON_Y + world_y / max(depth, 0.01))

# ══════════════════════════════════════════
#  BACKGROUND BUILDINGS  (deep / silhouette)
# ══════════════════════════════════════════
BG_BUILDINGS = []
bx = 0
while bx < W:
    bw = random.randint(40, 90)
    bh = random.randint(80, 220)
    BG_BUILDINGS.append({
        "x": bx, "w": bw, "h": bh,
        "color": (random.randint(6,14), random.randint(3,8), random.randint(12,26)),
        "windows": [
            (random.randint(4, bw-10), random.randint(6, bh-10))
            for _ in range(random.randint(3, 14))
            if random.random() > 0.3
        ],
        "win_col": random.choice([CYAN, PINK, YELLOW,
                                   (180,140,0), (0,140,160)]),
        "antenna": random.random() > 0.5,
        "ant_h":   random.randint(12, 40),
    })
    bx += bw + random.randint(0, 6)

# ══════════════════════════════════════════
#  FOREGROUND BUILDINGS  (close, sides)
# ══════════════════════════════════════════
# Left wall block
LEFT_WALL = {
    "x": 0, "y": int(H*0.10), "w": int(W*0.18), "h": H,
    "color": (10, 5, 20),
    "details": [   # (rx, ry, rw, rh, col)
        (8,  30, 60, 4,  CYAN),
        (8,  80, 45, 3,  PINK),
        (8, 130, 55, 4,  CYAN),
        (8, 180, 40, 3,  YELLOW),
        (8, 240, 50, 4,  PINK),
    ],
    "windows": [(random.randint(8,100), random.randint(20, 340))
                for _ in range(18)],
    "win_col": CYAN,
    "signage": [
        {"x":14,"y":60,  "w":90,"h":22,"col":PINK,  "text":"UPLOAD"},
        {"x":14,"y":160, "w":90,"h":18,"col":CYAN,  "text":"NEXUS"},
        {"x":14,"y":260, "w":90,"h":20,"col":YELLOW,"text":"DEPART"},
    ],
}

# Right wall block
RIGHT_WALL = {
    "x": int(W*0.82), "y": int(H*0.08), "w": int(W*0.18)+10, "h": H,
    "color": (8, 4, 18),
    "details": [
        (8,  40, 55, 4,  PINK),
        (8,  90, 65, 3,  CYAN),
        (8, 150, 48, 4,  YELLOW),
        (8, 200, 58, 3,  PINK),
        (8, 270, 44, 4,  CYAN),
    ],
    "windows": [(random.randint(8,120), random.randint(20, 340))
                for _ in range(16)],
    "win_col": PINK,
    "signage": [
        {"x":8,"y":50,  "w":88,"h":20,"col":CYAN,  "text":"STREAM"},
        {"x":8,"y":150, "w":88,"h":18,"col":PINK,  "text":"MERGE"},
        {"x":8,"y":260, "w":88,"h":22,"col":YELLOW,"text":"∞ VOID"},
    ],
}

# ══════════════════════════════════════════
#  STREET PROPS
# ══════════════════════════════════════════
# Dumpsters
DUMPSTERS = [
    {"x": 120, "y": GROUND_Y + 10, "w": 55, "h": 38,
     "col": (18,10,8), "stripe": YELLOW},
    {"x": 720, "y": GROUND_Y + 12, "w": 50, "h": 34,
     "col": (12, 8,18), "stripe": PINK},
]

# Street lights (perspective poles)
LIGHTS = [
    {"wx": -260, "depth": 1.4, "col": CYAN,  "on": True},
    {"wx":  260, "depth": 1.4, "col": PINK,  "on": True},
    {"wx": -180, "depth": 2.5, "col": YELLOW,"on": random.random()>0.3},
    {"wx":  180, "depth": 2.5, "col": CYAN,  "on": random.random()>0.3},
    {"wx": -100, "depth": 5.0, "col": PINK,  "on": random.random()>0.4},
    {"wx":  100, "depth": 5.0, "col": CYAN,  "on": random.random()>0.4},
]

# Wanted poster positions on walls
POSTERS = [
    {"x": 158, "y": 300, "w": 28, "h": 38, "col": (60,8,20),  "torn": True},
    {"x": 190, "y": 290, "w": 24, "h": 32, "col": (8,40,55),  "torn": False},
    {"x":W-158,"y": 310, "w": 26, "h": 36, "col": (55,30,5),  "torn": True},
]

# Ground debris / litter
DEBRIS = [(random.randint(155, W-155), GROUND_Y + random.randint(5,60),
           random.randint(3,12), random.randint(1,4),
           random.choice([(30,20,10),(20,30,10),(10,20,30),(40,20,20)]))
           for _ in range(40)]

# Puddles
PUDDLES = [
    {"x": 200, "y": GROUND_Y+28, "rx": 55, "ry": 10},
    {"x": 480, "y": GROUND_Y+18, "rx": 70, "ry":  8},
    {"x": 680, "y": GROUND_Y+35, "rx": 45, "ry":  9},
    {"x": 340, "y": GROUND_Y+50, "rx": 35, "ry":  7},
]

# ══════════════════════════════════════════
#  FIGURES  (shadowy people)
# ══════════════════════════════════════════
FIGURES = [
    {"x": 240, "speed":  0.18, "col": (20,10,30), "h": 52, "hooded": True},
    {"x": 590, "speed": -0.12, "col": (15, 8,25), "h": 46, "hooded": False},
    {"x": 420, "speed":  0.0,  "col": (18, 9,28), "h": 50, "hooded": True},
    {"x": 310, "speed": -0.22, "col": (22,10,32), "h": 44, "hooded": False},
]

# ══════════════════════════════════════════
#  RAIN
# ══════════════════════════════════════════
DROPS = [{"x": random.uniform(0,W), "y": random.uniform(0,H),
           "speed": random.uniform(7,14),
           "length": random.randint(10,24),
           "col": random.choice([DIM_CYAN, DIM_PINK, (25,15,40)])}
          for _ in range(220)]

# ══════════════════════════════════════════
#  NEON SIGNS  (flickering)
# ══════════════════════════════════════════
SIGNS = []
sign_pool = ["UPLOAD","MERGE ∞","DEPART","GHOST","NEXUS",
             "CIPHER","SYNTH","VOID.EXE","STREAM","2157"]
for _ in range(7):
    SIGNS.append({
        "x": random.randint(170, W-220),
        "y": random.randint(int(H*0.13), int(H*0.42)),
        "text": random.choice(sign_pool),
        "col":  random.choice([PINK,CYAN,YELLOW,(0,200,120),(180,0,200)]),
        "size": random.choice([11,13,15]),
        "on":   True,
        "timer":0,
        "rate": random.randint(60,500),
    })

pygame.font.init()
_fonts = {}
def font(size, bold=True):
    key = (size, bold)
    if key not in _fonts:
        _fonts[key] = pygame.font.SysFont("consolas", size, bold=bold)
    return _fonts[key]

# ══════════════════════════════════════════
#  SMOKE / STEAM PARTICLES
# ══════════════════════════════════════════
class Particle:
    def __init__(self, x, y, col):
        self.x  = x + random.uniform(-4, 4)
        self.y  = float(y)
        self.vx = random.uniform(-0.3, 0.3)
        self.vy = random.uniform(-0.6, -1.4)
        self.life   = random.randint(40, 110)
        self.maxlife= self.life
        self.r  = random.randint(4, 12)
        self.col= col

    def update(self):
        self.x  += self.vx
        self.y  += self.vy
        self.vx += random.uniform(-0.05, 0.05)
        self.life-= 1

    def draw(self, surf):
        if self.life <= 0: return
        t = self.life / self.maxlife
        a = int(55 * t)
        s = pygame.Surface((self.r*2, self.r*2), pygame.SRCALPHA)
        pygame.draw.circle(s, self.col+(a,), (self.r,self.r), self.r)
        surf.blit(s, (int(self.x)-self.r, int(self.y)-self.r))

STEAM_SOURCES = [
    {"x": 295, "y": GROUND_Y+5,  "col": (80,60,90),  "rate": 4},
    {"x": 580, "y": GROUND_Y+8,  "col": (60,70,90),  "rate": 6},
    {"x": 160, "y": GROUND_Y+15, "col": (70,55,80),  "rate": 8},
    {"x": 740, "y": GROUND_Y+10, "col": (65,65,85),  "rate": 5},
]
particles = []

# ══════════════════════════════════════════
#  SCANLINE SURFACE
# ══════════════════════════════════════════
scanline_surf = pygame.Surface((W, H), pygame.SRCALPHA)
for sy in range(0, H, 3):
    pygame.draw.line(scanline_surf, (0,0,0,22), (0,sy),(W,sy))

# ══════════════════════════════════════════
#  DRAW HELPERS
# ══════════════════════════════════════════
def draw_glow_circle(surf, col, cx, cy, r, alpha=60):
    for gr in range(r, 0, -5):
        a = int(alpha * (gr/r)**0.6)
        s = pygame.Surface((gr*2,gr*2), pygame.SRCALPHA)
        pygame.draw.circle(s, col+(a,), (gr,gr), gr)
        surf.blit(s, (cx-gr, cy-gr))

def draw_glow_rect(surf, col, rect, alpha=40, pad=6):
    s = pygame.Surface((rect[2]+pad*2, rect[3]+pad*2), pygame.SRCALPHA)
    s.fill(col+(alpha,))
    surf.blit(s, (rect[0]-pad, rect[1]-pad))

def draw_sign_text(surf, text, x, y, col, size, alpha=255):
    f   = font(size)
    txt = f.render(text, True, col)
    # glow
    g = f.render(text, True, col)
    g.set_alpha(60)
    surf.blit(g, (x-2, y+1))
    txt.set_alpha(alpha)
    surf.blit(txt, (x, y))

# window flicker state
win_states_bg   = {i:[random.random()>0.2 for _ in b["windows"]]
                   for i,b in enumerate(BG_BUILDINGS)}
win_states_left  = [random.random()>0.25 for _ in LEFT_WALL["windows"]]
win_states_right = [random.random()>0.25 for _ in RIGHT_WALL["windows"]]

tick = 0
running = True

while running:
    clock.tick(60)
    tick += 1

    for e in pygame.event.get():
        if e.type == pygame.QUIT: running = False
        if e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE: running = False

    # ── Sky ──
    for y in range(GROUND_Y+1):
        t = y / GROUND_Y
        r = int(SKY_TOP[0]+(SKY_BOT[0]-SKY_TOP[0])*t)
        g = int(SKY_TOP[1]+(SKY_BOT[1]-SKY_TOP[1])*t)
        b = int(SKY_TOP[2]+(SKY_BOT[2]-SKY_TOP[2])*t)
        pygame.draw.line(screen,(r,g,b),(0,y),(W,y))

    # ── Background buildings ──
    for i, b in enumerate(BG_BUILDINGS):
        by = HORIZON_Y - b["h"]
        pygame.draw.rect(screen, b["color"], (b["x"],by,b["w"],b["h"]))
        # edge glow
        pygame.draw.line(screen, b["win_col"],
                         (b["x"],by),(b["x"],by+b["h"]), 1)
        # windows
        for j,(wx,wy) in enumerate(b["windows"]):
            on = win_states_bg[i][j]
            if tick % random.randint(120,400) == (i*3+j)%80:
                win_states_bg[i][j] = not on; on = win_states_bg[i][j]
            col = b["win_col"] if on else (6,3,12)
            pygame.draw.rect(screen, col, (b["x"]+wx, by+wy, 4, 4))
        # antenna
        if b["antenna"]:
            ax = b["x"]+b["w"]//2
            pygame.draw.line(screen,(60,50,70),(ax,by),(ax,by-b["ant_h"]),1)
            bc = PINK if (tick//20)%2==0 else (20,5,15)
            pygame.draw.circle(screen, bc, (ax,by-b["ant_h"]),2)

    # ── Horizon glow ──
    hg = pygame.Surface((W, 60), pygame.SRCALPHA)
    for i in range(60):
        t   = i/60
        alp = int(70*(1-t))
        col = (int(PINK[0]*(1-t)+CYAN[0]*t),
               int(PINK[1]*(1-t)+CYAN[1]*t),
               int(PINK[2]*(1-t)+CYAN[2]*t))
        pygame.draw.line(hg, col+(alp,), (0,i),(W,i))
    screen.blit(hg,(0,HORIZON_Y-30))

    # ══ STREET PERSPECTIVE ══════════════════
    # Draw street surface with perspective lines
    street_surf = pygame.Surface((W, H-GROUND_Y), pygame.SRCALPHA)
    # Base street fill
    for y in range(H-GROUND_Y):
        t = y/(H-GROUND_Y)
        r = int(8+t*4); g=int(4+t*2); b2=int(14+t*6)
        pygame.draw.line(street_surf,(r,g,b2),(0,y),(W,y))

    # Perspective kerb lines
    for depth_step in range(1,12):
        d  = depth_step * 0.8
        sy = GROUND_Y + int((H-GROUND_Y)*(1-1/d)*0.9)
        if GROUND_Y <= sy < H:
            bright = max(0, 50-depth_step*4)
            col = (0, min(bright,50), min(bright+10,65))
            pygame.draw.line(screen, col, (0,sy),(W,sy),1)

    # Vanishing perspective lines on road
    for off in [-300,-200,-140,-80,-30, 30, 80,140,200,300]:
        ex = VP_X+off
        alp = max(0, 55-abs(off)//6)
        ls = pygame.Surface((W,H), pygame.SRCALPHA)
        pygame.draw.line(ls, DIM_CYAN+(alp,),(VP_X,GROUND_Y),(ex,H),1)
        screen.blit(ls,(0,0))

    # Road surface (flat)
    pygame.draw.rect(screen, GROUND_COL, (0,GROUND_Y,W,H-GROUND_Y))

    # Re-draw perspective lines on top of flat road
    for off in [-300,-200,-140,-80,-30,30,80,140,200,300]:
        ex = VP_X+off
        alp = max(0, 40-abs(off)//7)
        ls = pygame.Surface((W,H), pygame.SRCALPHA)
        pygame.draw.line(ls, DIM_CYAN+(alp,),(VP_X,GROUND_Y),(ex,H),1)
        screen.blit(ls,(0,0))

    # Horizontal road lines (distance markers)
    for i in range(1,8):
        ry = GROUND_Y + int((H-GROUND_Y)*(i/8)**0.6)
        a  = int(30*(i/8))
        rs = pygame.Surface((W,1),pygame.SRCALPHA)
        rs.fill(CYAN+(a,))
        screen.blit(rs,(0,ry))

    # ── Puddles (reflections) ──
    for p in PUDDLES:
        ps = pygame.Surface((p["rx"]*2, p["ry"]*2), pygame.SRCALPHA)
        pygame.draw.ellipse(ps, PUDDLE_COL+(180,), (0,0,p["rx"]*2,p["ry"]*2))
        screen.blit(ps,(p["x"]-p["rx"], p["y"]-p["ry"]))
        # Neon reflection shimmer in puddle
        shimmer_col = random.choice([CYAN,PINK,YELLOW])
        ripple = int(2*math.sin(tick*0.08 + p["x"]*0.02))
        rs2 = pygame.Surface((p["rx"]*2, p["ry"]*2), pygame.SRCALPHA)
        pygame.draw.ellipse(rs2, shimmer_col+(30,),
                            (ripple,1,p["rx"]*2-ripple*2,p["ry"]*2-2))
        screen.blit(rs2,(p["x"]-p["rx"],p["y"]-p["ry"]))

    # ── Left wall ──
    lw = LEFT_WALL
    pygame.draw.rect(screen, lw["color"], (lw["x"],lw["y"],lw["w"],lw["h"]))
    # Horizontal neon trim lines
    for rx,ry,rw,rh,rc in lw["details"]:
        pulse = 0.6+0.4*math.sin(tick*0.05+ry*0.03)
        col   = tuple(int(c*pulse) for c in rc)
        pygame.draw.rect(screen, col, (lw["x"]+rx,lw["y"]+ry,rw,rh))
        draw_glow_rect(screen, rc, (lw["x"]+rx,lw["y"]+ry,rw,rh), alpha=25)
    # Windows
    for j,(wx,wy) in enumerate(lw["windows"]):
        on = win_states_left[j]
        if tick % random.randint(100,350) == j%70:
            win_states_left[j] = not on; on = win_states_left[j]
        col = lw["win_col"] if on else (6,3,10)
        pygame.draw.rect(screen, col, (lw["x"]+wx,lw["y"]+wy,6,5))
    # Signage
    for sg in lw["signage"]:
        pulse = 0.7+0.3*math.sin(tick*0.07+sg["y"]*0.02)
        col   = tuple(int(c*pulse) for c in sg["col"])
        draw_glow_rect(screen, sg["col"],
                       (lw["x"]+sg["x"],lw["y"]+sg["y"],sg["w"],sg["h"]),
                       alpha=30, pad=4)
        pygame.draw.rect(screen, (0,0,0),
                         (lw["x"]+sg["x"],lw["y"]+sg["y"],sg["w"],sg["h"]))
        draw_sign_text(screen, sg["text"],
                       lw["x"]+sg["x"]+4, lw["y"]+sg["y"]+3,
                       col, 11)
    # Right edge glow of left wall
    eg = pygame.Surface((8,H), pygame.SRCALPHA)
    for ey in range(H):
        a = int(60*abs(math.sin(tick*0.03+ey*0.02)))
        eg.fill(CYAN+(a,),(0,ey,8,1))
    screen.blit(eg,(lw["x"]+lw["w"]-4,0))

    # ── Right wall ──
    rwall = RIGHT_WALL
    pygame.draw.rect(screen, rwall["color"],
                     (rwall["x"],rwall["y"],rwall["w"],rwall["h"]))
    for rx,ry,rw2,rh2,rc in rwall["details"]:
        pulse = 0.6+0.4*math.sin(tick*0.05+ry*0.03+1.2)
        col   = tuple(int(c*pulse) for c in rc)
        pygame.draw.rect(screen, col,
                         (rwall["x"]+rx,rwall["y"]+ry,rw2,rh2))
        draw_glow_rect(screen, rc,
                       (rwall["x"]+rx,rwall["y"]+ry,rw2,rh2), alpha=25)
    for j,(wx,wy) in enumerate(rwall["windows"]):
        on = win_states_right[j]
        if tick%random.randint(100,350)==j%65:
            win_states_right[j]=not on; on=win_states_right[j]
        col = rwall["win_col"] if on else (6,3,10)
        pygame.draw.rect(screen, col,
                         (rwall["x"]+wx,rwall["y"]+wy,6,5))
    for sg in rwall["signage"]:
        pulse = 0.7+0.3*math.sin(tick*0.07+sg["y"]*0.02+0.8)
        col   = tuple(int(c*pulse) for c in sg["col"])
        draw_glow_rect(screen, sg["col"],
                       (rwall["x"]+sg["x"],rwall["y"]+sg["y"],
                        sg["w"],sg["h"]), alpha=30, pad=4)
        pygame.draw.rect(screen,(0,0,0),
                         (rwall["x"]+sg["x"],rwall["y"]+sg["y"],
                          sg["w"],sg["h"]))
        draw_sign_text(screen, sg["text"],
                       rwall["x"]+sg["x"]+4, rwall["y"]+sg["y"]+3,
                       col, 11)
    # Left edge glow of right wall
    eg2 = pygame.Surface((8,H), pygame.SRCALPHA)
    for ey in range(H):
        a = int(60*abs(math.sin(tick*0.03+ey*0.02+0.9)))
        eg2.fill(PINK+(a,),(0,ey,8,1))
    screen.blit(eg2,(rwall["x"]-4,0))

    # ── Street lights ──
    for lt in LIGHTS:
        sx = persp_x(lt["wx"], lt["depth"])
        sy = persp_y(60,       lt["depth"])
        # pole
        gy = GROUND_Y + int((H-GROUND_Y)*0.1)
        pygame.draw.line(screen,(30,20,40),(sx,sy),(sx,gy),2)
        # arm
        arm = int(22/lt["depth"])
        pygame.draw.line(screen,(30,20,40),(sx,sy),(sx-arm,sy-arm//2),2)
        # lamp
        if lt["on"]:
            flicker = random.random() > 0.02
            if flicker:
                lc = lt["col"]
                draw_glow_circle(screen, lc, sx-arm, sy-arm//2,
                                 int(14/lt["depth"]), alpha=80)
                pygame.draw.circle(screen, lc,
                                   (sx-arm, sy-arm//2),
                                   max(2,int(4/lt["depth"])))
                # cone of light downward
                cone = pygame.Surface((W,H), pygame.SRCALPHA)
                cone_w = int(30/lt["depth"])
                cone_h = int(80/lt["depth"])
                pts = [(sx-arm, sy-arm//2),
                       (sx-arm-cone_w, sy-arm//2+cone_h),
                       (sx-arm+cone_w, sy-arm//2+cone_h)]
                pygame.draw.polygon(cone, lc+(18,), pts)
                screen.blit(cone,(0,0))
        else:
            pygame.draw.circle(screen,(20,10,20),(sx-arm,sy-arm//2),2)

    # ── Posters / graffiti on walls ──
    for po in POSTERS:
        pygame.draw.rect(screen, po["col"], (po["x"],po["y"],po["w"],po["h"]))
        if po["torn"]:
            # tear line
            tear_y = po["y"] + po["h"]//2 + random.randint(-3,3)
            pygame.draw.line(screen,(4,2,8),
                             (po["x"],tear_y),(po["x"]+po["w"],tear_y),2)
        # X through poster (wanted / defaced)
        pygame.draw.line(screen,(po["col"][0]//3,po["col"][1]//3,
                                 po["col"][2]//3+10),
                         (po["x"],po["y"]),
                         (po["x"]+po["w"],po["y"]+po["h"]),1)

    # ── Dumpsters ──
    for dm in DUMPSTERS:
        pygame.draw.rect(screen, dm["col"],
                         (dm["x"],dm["y"],dm["w"],dm["h"]))
        # lid
        pygame.draw.rect(screen,
                         tuple(min(255,c+10) for c in dm["col"]),
                         (dm["x"]-2, dm["y"]-5, dm["w"]+4, 8))
        # stripe
        for sx2 in range(dm["x"]+4, dm["x"]+dm["w"]-4, 12):
            pygame.draw.line(screen, dm["stripe"],
                             (sx2, dm["y"]+4),(sx2, dm["y"]+dm["h"]-4),2)
        # shadow beneath
        sh = pygame.Surface((dm["w"]+10,8), pygame.SRCALPHA)
        sh.fill(SHADOW+(100,))
        screen.blit(sh,(dm["x"]-5, dm["y"]+dm["h"]))

    # ── Ground debris ──
    for dx,dy,dw,dh,dc in DEBRIS:
        pygame.draw.rect(screen, dc, (dx,dy,dw,dh))

    # ── Shadowy figures ──
    for fig in FIGURES:
        fig["x"] += fig["speed"]
        if fig["x"] > W-155: fig["x"] = 155.0
        if fig["x"] < 155:   fig["x"] = float(W-155)
        fx, fy = int(fig["x"]), GROUND_Y
        fh = fig["h"]
        fc = fig["col"]
        # shadow on ground
        sh = pygame.Surface((30,8), pygame.SRCALPHA)
        pygame.draw.ellipse(sh, (0,0,0,80), (0,0,30,8))
        screen.blit(sh,(fx-15, fy+2))
        # legs
        leg_sway = int(3*math.sin(tick*0.12+fig["x"]*0.05)) if fig["speed"]!=0 else 0
        pygame.draw.line(screen,fc,(fx,fy-fh//3),(fx-5+leg_sway,fy),3)
        pygame.draw.line(screen,fc,(fx,fy-fh//3),(fx+5-leg_sway,fy),3)
        # body
        pygame.draw.rect(screen,fc,(fx-8,fy-fh,16,fh//2+4))
        # head
        if fig["hooded"]:
            pygame.draw.ellipse(screen,fc,(fx-7,fy-fh-10,14,14))
            # hood point
            pygame.draw.polygon(screen,fc,[(fx-7,fy-fh-4),
                                            (fx+7,fy-fh-4),
                                            (fx,fy-fh-18)])
        else:
            pygame.draw.circle(screen,fc,(fx,fy-fh-6),7)
        # occasional glowing eye
        if random.random()>0.996:
            eye_col = random.choice([CYAN,PINK,YELLOW])
            pygame.draw.circle(screen,eye_col,(fx+2,fy-fh-5),2)

    # ── Neon signs (mid-scene, floating) ──
    for sg in SIGNS:
        sg["timer"] += 1
        if sg["timer"] >= sg["rate"]:
            sg["on"]   = not sg["on"]
            sg["timer"]= 0
            sg["rate"] = random.randint(4,12) if not sg["on"] \
                         else random.randint(60,500)
        if not sg["on"]: continue
        pulse = 0.65+0.35*math.sin(tick*0.07+sg["x"]*0.01)
        col   = tuple(int(c*pulse) for c in sg["col"])
        draw_glow_rect(screen, sg["col"],
                       (sg["x"],sg["y"],
                        font(sg["size"]).size(sg["text"])[0],
                        sg["size"]+6), alpha=22, pad=5)
        draw_sign_text(screen,sg["text"],sg["x"],sg["y"],col,sg["size"])

    # ── Steam / smoke particles ──
    for src in STEAM_SOURCES:
        if tick % src["rate"] == 0:
            particles.append(Particle(src["x"],src["y"],src["col"]))
    ps = pygame.Surface((W,H), pygame.SRCALPHA)
    for p in particles:
        p.update(); p.draw(ps)
    particles[:] = [p for p in particles if p.life > 0]
    screen.blit(ps,(0,0))

    # ── Rain ──
    rs = pygame.Surface((W,H), pygame.SRCALPHA)
    for d in DROPS:
        d["y"] += d["speed"]
        d["x"] += d["speed"]*0.14
        if d["y"] > H:
            d["y"] = random.uniform(-20,0)
            d["x"] = random.uniform(0,W)
        ex = int(d["x"]+d["length"]*0.14)
        ey = int(d["y"]+d["length"])
        pygame.draw.line(rs, d["col"]+(50,),
                         (int(d["x"]),int(d["y"])),(ex,ey),1)
    screen.blit(rs,(0,0))

    # ── Scanlines ──
    screen.blit(scanline_surf,(0,0))

    # ── Fog / atmosphere vignette ──
    fog = pygame.Surface((W,H), pygame.SRCALPHA)
    # Corners darker
    for corner_x, corner_y in [(0,0),(W,0),(0,H),(W,H)]:
        for r2 in range(180,0,-20):
            a2 = int(40*(1-r2/180))
            pygame.draw.circle(fog,(2,1,8,a2),(corner_x,corner_y),r2)
    screen.blit(fog,(0,0))

    # ── HUD ──
    hud_f = font(11)
    for txt, col2, y2 in [
        ("// STREET LEVEL — NEON DISTRICT — 2157 //", CYAN,  8),
        ("DANGER ZONE  |  RAIN HEAVY  |  VIS: LOW",  PINK, 24),
    ]:
        s = hud_f.render(txt, True, col2)
        s.set_alpha(110)
        screen.blit(s,(8,y2))

    pygame.display.flip()

pygame.quit()



# loading saved game

print("Now we must fight!")

import pygame
import math
import random
import numpy as np

pygame.init()
pygame.mixer.init(frequency=44100, size=-16, channels=1, buffer=512)

W, H = 600, 600
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("...")
clock = pygame.time.Clock()

# ══════════════════════════════════════════
#  GENERATE "YAHHHHH" SCREAM SOUND
# ══════════════════════════════════════════
def make_scream():
    rate     = 44100
    duration = 2.8
    n        = int(rate * duration)
    samples  = np.zeros(n, dtype=np.float32)

    for i in range(n):
        t   = i / rate
        env = 1.0

        # Fast attack
        if t < 0.05:
            env = t / 0.05
        # Sustain with slight decay
        elif t < 2.2:
            env = 1.0 - (t - 0.05) * 0.18
        # Fade out
        else:
            env = max(0.0, 1.0 - (t - 2.2) / 0.6) * 0.6

        env = max(0.0, env)

        # ── Core scream frequencies ──
        # Fundamental (rough male yell ~250 Hz dropping)
        f0  = 260 - t * 30
        y0  = math.sin(2*math.pi*f0*t)

        # 2nd harmonic (adds richness)
        y1  = 0.6  * math.sin(2*math.pi*f0*2*t)

        # 3rd harmonic (harsh edge)
        y2  = 0.4  * math.sin(2*math.pi*f0*3*t)

        # 4th harmonic (screamy brightness)
        y3  = 0.3  * math.sin(2*math.pi*f0*4*t + 0.5)

        # ── Throat / vocal roughness (sub-harmonics) ──
        rough = 0.25 * math.sin(2*math.pi*(f0*0.5)*t)

        # ── Noise burst (breathiness / air) ──
        noise = (random.random()*2-1) * 0.18

        # ── Vibrato (natural voice wobble) ──
        vibrato = math.sin(2*math.pi*6.5*t) * 0.02
        vib_y   = math.sin(2*math.pi*(f0*(1+vibrato))*t) * 0.5

        # ── Formant resonance peak (vowel "AH" ~800 Hz) ──
        formant = 0.35 * math.sin(2*math.pi*820*t) * math.exp(-t*1.5)

        # Mix
        sample = (y0 + y1 + y2 + y3 + rough + noise + vib_y + formant) * env
        samples[i] = sample

    # Normalise
    peak = np.max(np.abs(samples))
    if peak > 0:
        samples /= peak

    # Convert to 16-bit PCM
    pcm = (samples * 32000).astype(np.int16)
    sound = pygame.sndarray.make_sound(pcm)
    return sound

# scream = make_scream()

# ══════════════════════════════════════════
#  COLOUR PALETTE
# ══════════════════════════════════════════
BLACK   = (0,   0,   0)
WHITE   = (255, 255, 255)
RED     = (200,  10,  10)
DARK_R  = (100,   0,   0)
SKIN    = (180, 120,  80)
SKIN_D  = (120,  70,  40)
SKIN_DD = ( 70,  30,  10)
TEETH   = (230, 220, 190)
TONGUE  = (180,  40,  60)
EYE_W   = (220, 210, 200)
VEIN    = (140,  20,  20)
PINK    = (200,  80, 120)
PURPLE  = ( 80,  20, 120)
DARK    = (  8,   0,  12)

# ══════════════════════════════════════════
#  STATE
# ══════════════════════════════════════════
# Phases: "dark" → "approach" → "SCREAM"
phase         = "dark"
dark_timer    = 0      # frames of darkness at start
approach_t    = 0.0    # 0→1, face zooms from far
DARK_HOLD     = 90     # 1.5 s of pure black
APPROACH_DUR  = 120    # 2 s to zoom in
scream_played = False
scream_timer  = 0

# ── Shake state ──
shake_x = 0
shake_y = 0

# ══════════════════════════════════════════
#  DRAW HELPERS
# ══════════════════════════════════════════
def draw_ellipse_aa(surf, col, rect):
    pygame.draw.ellipse(surf, col, rect)

def glow(surf, col, cx, cy, r, alpha=60, steps=6):
    s = pygame.Surface((r*2+4, r*2+4), pygame.SRCALPHA)
    for i in range(steps, 0, -1):
        a = int(alpha * (i/steps)**0.5)
        gr = int(r * i / steps)
        pygame.draw.circle(s, col+(a,), (r+2, r+2), gr)
    surf.blit(s, (cx-r-2, cy-r-2))

def lerp(a, b, t):
    return a + (b-a)*t

def ease_out(t):
    return 1 - (1-t)**3

def ease_in(t):
    return t**2

# ══════════════════════════════════════════
#  FACE DRAW
# ══════════════════════════════════════════
def draw_face(surf, cx, cy, scale, scream_phase, tick):
    """
    Draw the scary face centred at (cx,cy) with given scale.
    scream_phase: 0=approaching (eyes open slight), 1=full scream
    """
    sc = scale

    # ── Background glow (aura) ──
    if scream_phase > 0.3:
        aura_r = int(280*sc)
        aura_s = pygame.Surface((aura_r*2, aura_r*2), pygame.SRCALPHA)
        for r2 in range(aura_r, 0, -20):
            a = int(60*(r2/aura_r)**0.4 * scream_phase)
            pygame.draw.circle(aura_s, RED+(a,), (aura_r,aura_r), r2)
        surf.blit(aura_s, (cx-aura_r, cy-aura_r))

    # ── Neck ──
    neck_w = int(90*sc)
    neck_h = int(120*sc)
    neck_x = cx - neck_w//2
    neck_y = cy + int(120*sc)
    pygame.draw.rect(surf, SKIN_D,
                     (neck_x, neck_y, neck_w, neck_h))
    # Neck shadow lines
    for i in range(3):
        lx = neck_x + int((15+i*20)*sc)
        pygame.draw.line(surf, SKIN_DD,
                         (lx, neck_y), (lx, neck_y+neck_h), max(1,int(2*sc)))

    # ── Head shape ──
    head_w = int(240*sc)
    head_h = int(280*sc)
    # Shadow / depth
    pygame.draw.ellipse(surf, SKIN_DD,
                        (cx-head_w//2+int(8*sc),
                         cy-head_h//2+int(10*sc),
                         head_w, head_h))
    # Base skin
    pygame.draw.ellipse(surf, SKIN,
                        (cx-head_w//2, cy-head_h//2, head_w, head_h))

    # ── Forehead veins ──
    if scream_phase > 0.5:
        vein_alpha = min(255, int((scream_phase-0.5)*2*200))
        vs = pygame.Surface((W,H), pygame.SRCALPHA)
        # Left vein
        for j in range(4):
            vx = cx - int((60-j*8)*sc)
            vy = cy - int((100-j*15)*sc)
            vex = vx - int(20*sc) + j*int(5*sc)
            vey = vy - int(30*sc)
            pygame.draw.line(vs, VEIN+(vein_alpha,),
                             (vx,vy),(vex,vey), max(1,int((3-j)*sc)))
        # Right vein
        for j in range(3):
            vx = cx + int((50-j*6)*sc)
            vy = cy - int((80-j*12)*sc)
            vex = vx + int(15*sc)
            vey = vy - int(25*sc)
            pygame.draw.line(vs, VEIN+(vein_alpha,),
                             (vx,vy),(vex,vey), max(1,int((2-j//2)*sc)))
        surf.blit(vs,(0,0))

    # ── Brow / forehead shadow ──
    brow_rect = (cx-int(110*sc), cy-int(130*sc),
                 int(220*sc), int(50*sc))
    pygame.draw.ellipse(surf, SKIN_D, brow_rect)

    # ── Eyebrows (angry — slanted inward) ──
    brow_thick = max(2, int(8*sc))
    # Left brow
    pygame.draw.line(surf, SKIN_DD,
                     (cx-int(95*sc), cy-int(105*sc)),
                     (cx-int(25*sc), cy-int(120*sc)),
                     brow_thick)
    # Right brow
    pygame.draw.line(surf, SKIN_DD,
                     (cx+int(25*sc), cy-int(120*sc)),
                     (cx+int(95*sc), cy-int(105*sc)),
                     brow_thick)

    # ── Eyes ──
    eye_open  = 0.5 + scream_phase*0.5   # 0.5→1.0 as scream grows
    eye_rx    = int(38*sc)
    eye_ry    = int(28*sc * eye_open)
    l_ex, l_ey = cx-int(65*sc), cy-int(60*sc)
    r_ex, r_ey = cx+int(65*sc), cy-int(60*sc)

    # Eye whites
    pygame.draw.ellipse(surf, EYE_W,
                        (l_ex-eye_rx, l_ey-eye_ry, eye_rx*2, eye_ry*2))
    pygame.draw.ellipse(surf, EYE_W,
                        (r_ex-eye_rx, r_ey-eye_ry, eye_rx*2, eye_ry*2))

    # Bloodshot veins in whites
    if scream_phase > 0.4:
        for eye_cx, eye_cy in [(l_ex,l_ey),(r_ex,r_ey)]:
            vs2 = pygame.Surface((W,H), pygame.SRCALPHA)
            for _ in range(5):
                angle = random.uniform(0, math.pi*2)
                vlen  = random.uniform(0.4,0.9)
                ex2   = eye_cx + int(math.cos(angle)*eye_rx*vlen)
                ey2   = eye_cy + int(math.sin(angle)*eye_ry*vlen*0.6)
                pygame.draw.line(vs2, (180,0,0,120),
                                 (eye_cx,eye_cy),(ex2,ey2),1)
            surf.blit(vs2,(0,0))

    # Irises (dark, wide with fear/rage)
    iris_r = int(18*sc)
    # Pupils dilate with scream
    pupil_r = int(iris_r*(0.5 + scream_phase*0.4))
    # Slight downward look
    look_dy = int(4*sc)

    # Left iris + pupil
    pygame.draw.circle(surf, DARK_R,
                       (l_ex, l_ey+look_dy), iris_r)
    pygame.draw.circle(surf, BLACK,
                       (l_ex, l_ey+look_dy), pupil_r)
    # Right iris + pupil
    pygame.draw.circle(surf, DARK_R,
                       (r_ex, r_ey+look_dy), iris_r)
    pygame.draw.circle(surf, BLACK,
                       (r_ex, r_ey+look_dy), pupil_r)

    # Eye shine
    shine_r = max(1, int(4*sc))
    pygame.draw.circle(surf, WHITE,
                       (l_ex-int(8*sc), l_ey+look_dy-int(6*sc)), shine_r)
    pygame.draw.circle(surf, WHITE,
                       (r_ex-int(8*sc), r_ey+look_dy-int(6*sc)), shine_r)

    # Eye outline
    pygame.draw.ellipse(surf, SKIN_DD,
                        (l_ex-eye_rx, l_ey-eye_ry, eye_rx*2, eye_ry*2), 2)
    pygame.draw.ellipse(surf, SKIN_DD,
                        (r_ex-eye_rx, r_ey-eye_ry, eye_rx*2, eye_ry*2), 2)

    # ── Nose ──
    nose_cx = cx
    nose_cy = cy + int(10*sc)
    nose_w  = int(40*sc)
    nose_h  = int(55*sc)
    # Bridge
    pygame.draw.line(surf, SKIN_D,
                     (nose_cx-int(5*sc), cy-int(40*sc)),
                     (nose_cx-int(10*sc), nose_cy+int(5*sc)),
                     max(1,int(4*sc)))
    pygame.draw.line(surf, SKIN_D,
                     (nose_cx+int(5*sc), cy-int(40*sc)),
                     (nose_cx+int(10*sc), nose_cy+int(5*sc)),
                     max(1,int(4*sc)))
    # Nostrils
    pygame.draw.ellipse(surf, SKIN_DD,
                        (nose_cx-nose_w//2, nose_cy, nose_w, int(24*sc)))
    pygame.draw.ellipse(surf, BLACK,
                        (nose_cx-int(16*sc), nose_cy+int(4*sc),
                         int(14*sc), int(12*sc)))
    pygame.draw.ellipse(surf, BLACK,
                        (nose_cx+int(2*sc), nose_cy+int(4*sc),
                         int(14*sc), int(12*sc)))

    # ── Mouth ──
    # Opens wide with scream
    mouth_cx = cx
    mouth_cy = cy + int(80*sc)
    mouth_w  = int((80 + scream_phase*100)*sc)
    mouth_h  = int((20 + scream_phase*90)*sc)

    # Outer lip shadow
    pygame.draw.ellipse(surf, SKIN_DD,
                        (mouth_cx-mouth_w//2-int(4*sc),
                         mouth_cy-mouth_h//2-int(4*sc),
                         mouth_w+int(8*sc), mouth_h+int(8*sc)))
    # Dark mouth interior
    pygame.draw.ellipse(surf, BLACK,
                        (mouth_cx-mouth_w//2, mouth_cy-mouth_h//2,
                         mouth_w, mouth_h))

    if scream_phase > 0.2:
        # ── Throat (dark red) ──
        throat_w = int(mouth_w * 0.5)
        throat_h = int(mouth_h * 0.6)
        pygame.draw.ellipse(surf, (40,5,5),
                            (mouth_cx-throat_w//2,
                             mouth_cy-throat_h//2+int(mouth_h*0.2),
                             throat_w, throat_h))
        # Uvula
        if scream_phase > 0.5:
            uv_x = mouth_cx
            uv_y = mouth_cy + int(mouth_h*0.15)
            pygame.draw.line(surf, (120,20,30),
                             (uv_x, uv_y-int(10*sc)),
                             (uv_x, uv_y+int(8*sc)),
                             max(1,int(4*sc)))
            pygame.draw.circle(surf, (150,30,40),
                               (uv_x, uv_y+int(8*sc)),
                               max(2,int(5*sc)))

        # ── Tongue ──
        tongue_w = int(mouth_w*0.55)
        tongue_h = int(mouth_h*0.45)
        tongue_y = mouth_cy + int(mouth_h*0.1)
        pygame.draw.ellipse(surf, TONGUE,
                            (mouth_cx-tongue_w//2, tongue_y,
                             tongue_w, tongue_h))
        # Tongue groove
        pygame.draw.line(surf, (140,20,40),
                         (mouth_cx, tongue_y+int(4*sc)),
                         (mouth_cx, tongue_y+tongue_h-int(4*sc)),
                         max(1,int(3*sc)))

        # ── Teeth (upper) ──
        num_teeth  = 6
        tooth_w    = int(mouth_w * 0.75 / num_teeth)
        tooth_h    = int(mouth_h * 0.30)
        teeth_start = mouth_cx - int(mouth_w*0.75)//2
        for t_idx in range(num_teeth):
            tx = teeth_start + t_idx*(tooth_w+int(2*sc))
            ty = mouth_cy - mouth_h//2
            pygame.draw.rect(surf, TEETH,
                             (tx, ty, tooth_w-int(1*sc), tooth_h))
            # Tooth shadow
            pygame.draw.line(surf, (180,170,140),
                             (tx+tooth_w-int(2*sc), ty),
                             (tx+tooth_w-int(2*sc), ty+tooth_h),
                             max(1,int(2*sc)))

        # ── Lower teeth ──
        num_lower = 5
        ltooth_w  = int(mouth_w * 0.65 / num_lower)
        ltooth_h  = int(mouth_h * 0.22)
        lteeth_start = mouth_cx - int(mouth_w*0.65)//2
        for t_idx in range(num_lower):
            tx = lteeth_start + t_idx*(ltooth_w+int(2*sc))
            ty = mouth_cy + mouth_h//2 - ltooth_h
            pygame.draw.rect(surf, TEETH,
                             (tx, ty, ltooth_w-int(1*sc), ltooth_h))

    # Upper lip
    pygame.draw.arc(surf, SKIN_D,
                    (mouth_cx-mouth_w//2, mouth_cy-mouth_h//2-int(8*sc),
                     mouth_w, int(24*sc)),
                    0, math.pi, max(1,int(5*sc)))
    # Lower lip
    pygame.draw.arc(surf, SKIN,
                    (mouth_cx-mouth_w//2, mouth_cy+mouth_h//2-int(12*sc),
                     mouth_w, int(24*sc)),
                    math.pi, math.pi*2, max(1,int(6*sc)))

    # ── Cheek lines (tension) ──
    if scream_phase > 0.3:
        tension = scream_phase
        pygame.draw.line(surf, SKIN_DD,
                         (cx-int(100*sc), cy+int(20*sc)),
                         (cx-int(60*sc),  cy+int(60*sc)),
                         max(1,int(2*sc*tension)))
        pygame.draw.line(surf, SKIN_DD,
                         (cx+int(100*sc), cy+int(20*sc)),
                         (cx+int(60*sc),  cy+int(60*sc)),
                         max(1,int(2*sc*tension)))

    # ── Ear (left) ──
    ear_rx = int(22*sc)
    ear_ry = int(35*sc)
    pygame.draw.ellipse(surf, SKIN,
                        (cx-head_w//2-ear_rx+int(10*sc),
                         cy-ear_ry+int(10*sc),
                         ear_rx*2, ear_ry*2))
    pygame.draw.ellipse(surf, SKIN_D,
                        (cx-head_w//2-ear_rx+int(18*sc),
                         cy-ear_ry+int(16*sc),
                         int(ear_rx*0.9), int(ear_ry*1.1)))

    # ── Ear (right) ──
    pygame.draw.ellipse(surf, SKIN,
                        (cx+head_w//2-ear_rx-int(10*sc),
                         cy-ear_ry+int(10*sc),
                         ear_rx*2, ear_ry*2))
    pygame.draw.ellipse(surf, SKIN_D,
                        (cx+head_w//2-ear_rx-int(5*sc),
                         cy-ear_ry+int(16*sc),
                         int(ear_rx*0.9), int(ear_ry*1.1)))

    # ── Hair (messy / wild) ──
    hair_col = (20, 12, 8)
    # Top mass
    pygame.draw.ellipse(surf, hair_col,
                        (cx-int(130*sc), cy-int(160*sc),
                         int(260*sc), int(110*sc)))
    # Wild strands sticking up
    strand_data = [(-80,6),(-50,18),(-20,22),(10,20),
                   (40,15),(70,8),(-100,-2),(100,-4)]
    for sdx, sdy in strand_data:
        sx1 = cx + int(sdx*sc)
        sy1 = cy - int(140*sc)
        # Strand end wobbles with scream
        wobble = int(scream_phase * random.choice([-1,1]) * 12*sc)
        sx2 = sx1 + wobble
        sy2 = sy1 - int((30+abs(sdx)*0.4)*sc) + int(sdy*sc)
        pygame.draw.line(surf, hair_col, (sx1,sy1),(sx2,sy2),
                         max(1,int(4*sc)))

    # ── Face outline ──
    pygame.draw.ellipse(surf, SKIN_DD,
                        (cx-head_w//2, cy-head_h//2, head_w, head_h),
                        max(1,int(3*sc)))

# ══════════════════════════════════════════
#  SCREAM TEXT
# ══════════════════════════════════════════
pygame.font.init()
_fonts = {}
def get_font(size):
    if size not in _fonts:
        _fonts[size] = pygame.font.SysFont("impact", size, bold=True)
    return _fonts[size]

def draw_scream_text(surf, alpha, tick, scale):
    text  = "YAHHHHH!!!"
    # Wobble size
    size  = int(60 + scale*30 + math.sin(tick*0.3)*8)
    f     = get_font(size)
    # Red glow pass
    for ox, oy in [(-3,0),(3,0),(0,-3),(0,3),(-2,-2),(2,2)]:
        gs = f.render(text, True, (180,0,0))
        gs.set_alpha(max(0,alpha-120))
        rx = W//2 - gs.get_width()//2
        ry = int(H*0.88) - gs.get_height()//2
        surf.blit(gs,(rx+ox, ry+oy))
    # Main white text
    ts = f.render(text, True, WHITE)
    ts.set_alpha(alpha)
    rx = W//2 - ts.get_width()//2
    ry = int(H*0.88) - ts.get_height()//2
    # Shake text independently
    tx2 = rx + random.randint(-int(scale*6),int(scale*6))
    ty2 = ry + random.randint(-int(scale*4),int(scale*4))
    surf.blit(ts,(tx2,ty2))

# ══════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════
tick = 0
running = True

while running:
    clock.tick(60)
    tick += 1

    for e in pygame.event.get():
        if e.type == pygame.QUIT: running = False
        if e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE:
            running = False
        if e.type == pygame.KEYDOWN and e.key == pygame.K_SPACE:
            # Restart
            phase         = "dark"
            dark_timer    = 0
            approach_t    = 0.0
            scream_played = False
            scream_timer  = 0
            pygame.mixer.stop()

    # ── Phase logic ──
    if phase == "dark":
        dark_timer += 1
        if dark_timer >= DARK_HOLD:
            phase = "approach"

    elif phase == "approach":
        approach_t = min(1.0, approach_t + 1.0/APPROACH_DUR)
        if approach_t >= 1.0:
            phase = "SCREAM"
            scream_timer = 0

    elif phase == "SCREAM":
        scream_timer += 1
        #if not scream_played:
           # scream.play()
          #  scream_played = True

    # ══════════════════════════════════════
    #  RENDER
    # ══════════════════════════════════════
    screen.fill(BLACK)

    if phase == "dark":
        # Pure black with very faint breathing sound buildup
        # Just a tiny red hint in the centre at the very end
        if dark_timer > 60:
            hint_a = int((dark_timer-60)/30 * 15)
            hs = pygame.Surface((W,H), pygame.SRCALPHA)
            pygame.draw.circle(hs, RED+(hint_a,), (W//2,H//2), 80)
            screen.blit(hs,(0,0))

    elif phase == "approach":
        t         = ease_out(approach_t)
        scale     = 0.08 + t*0.92          # grows from tiny to full
        scream_ph = t * 0.6                 # face starts neutral, opens slightly
        cx        = W//2
        cy        = int(H*0.44)

        # Background reddens
        bg_r = int(t * 30)
        screen.fill((bg_r, 0, bg_r//3))

        # Shake increases as it gets closer
        if t > 0.7:
            shake_x = random.randint(-int((t-0.7)*20), int((t-0.7)*20))
            shake_y = random.randint(-int((t-0.7)*14), int((t-0.7)*14))
        else:
            shake_x = shake_y = 0

        draw_face(screen, cx+shake_x, cy+shake_y, scale, scream_ph, tick)

        # Vignette
        vig = pygame.Surface((W,H), pygame.SRCALPHA)
        for r3 in range(300,0,-30):
            a3 = int(80*(1-r3/300)*t)
            pygame.draw.circle(vig, BLACK+(a3,),(W//2,H//2),r3)
        # Invert — darken edges
        vig2 = pygame.Surface((W,H), pygame.SRCALPHA)
        vig2.fill((0,0,0,0))
        for r3 in range(W,int(W*0.3),-20):
            a3 = int(120*(r3-W*0.3)/(W*0.7)*t*0.6)
            pygame.draw.circle(vig2,BLACK+(a3,),(W//2,H//2),r3,20)
        screen.blit(vig2,(0,0))

    elif phase == "SCREAM":
        # Full scream — maximum chaos
        st      = min(1.0, scream_timer / 40.0)   # 0→1 over first 40 frames
        scale   = 1.0 + st*0.08*math.sin(tick*0.4)
        cx      = W//2
        cy      = int(H*0.44)

        # Pulsing red/black background
        pulse   = 0.5 + 0.5*math.sin(tick*0.25)
        bg_r    = int(15 + pulse*45)
        bg_g    = 0
        bg_b    = int(pulse*12)
        screen.fill((bg_r, bg_g, bg_b))

        # Heavy screen shake
        shake_x = random.randint(-14,14)
        shake_y = random.randint(-10,10)

        draw_face(screen, cx+shake_x, cy+shake_y, scale, 1.0, tick)

        # Radial red flash pulses
        for r4 in range(260, 0, -22):
            a4 = int(80*(1-r4/260)*pulse*0.8)
            fs = pygame.Surface((r4*2,r4*2), pygame.SRCALPHA)
            pygame.draw.circle(fs, RED+(a4,),(r4,r4),r4)
            screen.blit(fs,(cx-r4,cy-r4))

        # YAHHHH text
        text_a = min(255, scream_timer*12)
        draw_scream_text(screen, text_a, tick, st)

        # Chromatic aberration edges
        if st > 0.5:
            ca = pygame.Surface((W,H), pygame.SRCALPHA)
            for edge_r in range(W//2, int(W*0.3), -15):
                a5 = int(40*(1-edge_r/(W*0.5))*st)
                pygame.draw.circle(ca, RED+(a5,),(W//2,H//2),edge_r,15)
            screen.blit(ca,(0,0))

        # Scanlines for intensity
        for sy in range(0,H,3):
            pygame.draw.line(screen,(0,0,0),(0,sy),(W,sy),1)

        # After 5 seconds, loop back to dark
        if scream_timer > 300:
            phase         = "dark"
            dark_timer    = 0
            approach_t    = 0.0
            scream_played = False
            scream_timer  = 0

    # ── HUD ──
    f_hud = get_font(12)
    hint  = f_hud.render("SPACE to restart  ·  ESC to quit", True, (40,20,20))
    hint.set_alpha(60)
    screen.blit(hint,(W//2-hint.get_width()//2, H-24))

    pygame.display.flip()

pygame.quit()

# creates random enemies to fight and starts battle
import random

# ── Constants (set these to match your game) ──
W        = 900
GROUND_Y = int(500 * 0.62)

# ══════════════════════════════════════════════
#  ENEMY TEMPLATES
#  Each template defines a character archetype.
#  Stats are then randomised within ranges.
# ══════════════════════════════════════════════
ENEMY_TYPES = {

    "Street Thug": {
        "col_range":  [(180,60,60),  (200,80,40),  (160,40,60)],
        "hp_range":   (18, 35),
        "damage_range": (3, 7),
        "speed_range":  (0.6, 1.4),
        "size":       (18, 22),
        "xp_reward":  (10, 25),
        "gold_reward":(5,  20),
        "loot_table": ["Stim Pack", "Broken Blade", None, None],
        "abilities":  ["brawl"],
        "flavor":     "A scarred local enforcer. Hits hard, thinks slow.",
    },

    "Neon Rat": {
        "col_range":  [(80,200,80),  (60,220,100), (100,180,60)],
        "hp_range":   (8,  18),
        "damage_range": (1, 4),
        "speed_range":  (1.8, 3.2),
        "size":       (10, 14),
        "xp_reward":  (5,  15),
        "gold_reward":(1,  8),
        "loot_table": [None, None, None, "Scrap Wire"],
        "abilities":  ["dodge"],
        "flavor":     "Mutated vermin. Fast and annoying.",
    },

    "Corp Drone": {
        "col_range":  [(60,120,200), (40,100,220), (80,140,180)],
        "hp_range":   (25, 50),
        "damage_range": (5, 10),
        "speed_range":  (0.4, 0.9),
        "size":       (20, 26),
        "xp_reward":  (20, 40),
        "gold_reward":(15, 35),
        "loot_table": ["EMP Chip", "Corp Badge", "Stim Pack", None],
        "abilities":  ["shield_pulse", "brawl"],
        "flavor":     "Corporate security bot. Slow but armoured.",
    },

    "Ghost Runner": {
        "col_range":  [(180,0,200),  (160,0,220), (200,20,180)],
        "hp_range":   (15, 30),
        "damage_range": (8, 15),
        "speed_range":  (1.5, 2.8),
        "size":       (14, 18),
        "xp_reward":  (30, 55),
        "gold_reward":(20, 50),
        "loot_table": ["Ghost Blade", "Camo Cloak", None, None],
        "abilities":  ["dodge", "crit_strike"],
        "flavor":     "A hired blade. You may not see the first hit.",
    },

    "Sewer Mutant": {
        "col_range":  [(80,140,40),  (60,120,30),  (100,160,50)],
        "hp_range":   (40, 80),
        "damage_range": (6, 12),
        "speed_range":  (0.3, 0.8),
        "size":       (26, 34),
        "xp_reward":  (35, 60),
        "gold_reward":(5,  15),
        "loot_table": ["Mutant Gland", None, None, None],
        "abilities":  ["brawl", "poison_bite"],
        "flavor":     "Something the sewers made. Tough and wrong.",
    },

    "Rogue AI Shell": {
        "col_range":  [(0,220,220),  (0,200,240),  (20,240,200)],
        "hp_range":   (60, 100),
        "damage_range": (10, 18),
        "speed_range":  (0.8, 1.6),
        "size":       (22, 30),
        "xp_reward":  (60, 100),
        "gold_reward":(40, 80),
        "loot_table": ["AI Core", "EMP Chip", "Corp Badge", "Ghost Blade"],
        "abilities":  ["shield_pulse", "crit_strike", "brawl"],
        "flavor":     "A discarded machine that learned to hate. Boss tier.",
    },
}

# ══════════════════════════════════════════════
#  ABILITY DEFINITIONS
# ══════════════════════════════════════════════
ABILITIES = {
    "brawl": {
        "name":        "Brawl",
        "description": "A wild swing. Deals normal damage.",
        "damage_mult": 1.0,
        "effect":      None,
    },
    "dodge": {
        "name":        "Dodge",
        "description": "Sidesteps your attack — takes no damage this turn.",
        "damage_mult": 0.0,
        "effect":      "miss",
    },
    "crit_strike": {
        "name":        "Critical Strike",
        "description": "A precise hit. Deals double damage.",
        "damage_mult": 2.0,
        "effect":      "crit",
    },
    "shield_pulse": {
        "name":        "Shield Pulse",
        "description": "Activates armour shield — blocks 50% of incoming damage.",
        "damage_mult": 0.5,
        "effect":      "shield",
    },
    "poison_bite": {
        "name":        "Poison Bite",
        "description": "Deals damage and poisons you for 3 turns.",
        "damage_mult": 0.8,
        "effect":      "poison",
    },
}

# ══════════════════════════════════════════════
#  PLAYER ACTIONS
# ══════════════════════════════════════════════
PLAYER_ACTIONS = {
    "attack": {
        "name":        "Attack",
        "description": "Standard strike.",
        "damage_mult": 1.0,
        "stamina_cost": 5,
    },
    "heavy": {
        "name":        "Heavy Strike",
        "description": "Costs more stamina but hits for 1.8× damage.",
        "damage_mult": 1.8,
        "stamina_cost": 14,
    },
    "dodge": {
        "name":        "Dodge",
        "description": "Attempt to dodge enemy's next attack (70% chance).",
        "damage_mult": 0.0,
        "stamina_cost": 8,
    },
    "use_item": {
        "name":        "Use Item",
        "description": "Use a consumable from your inventory.",
        "damage_mult": 0.0,
        "stamina_cost": 0,
    },
    "flee": {
        "name":        "Flee",
        "description": "Attempt to escape (60% success chance).",
        "damage_mult": 0.0,
        "stamina_cost": 10,
    },
}

# ══════════════════════════════════════════════
#  GENERATE ENEMY
# ══════════════════════════════════════════════
def make_enemy(enemy_type=None, level=1):
    """
    Generate a randomised enemy dict.
    Pass enemy_type=None for a random pick,
    or enemy_type="Street Thug" etc. for a specific one.
    level scales stats upward.
    """
    if enemy_type is None:
        enemy_type = random.choice(list(ENEMY_TYPES.keys()))

    t   = ENEMY_TYPES[enemy_type]
    lvl = max(1, level)

    # Randomise base stats within template ranges
    base_hp  = random.randint(*t["hp_range"])
    base_dmg = random.randint(*t["damage_range"])
    speed    = round(random.uniform(*t["speed_range"]), 2)
    size     = random.randint(*t["size"])
    col      = random.choice(t["col_range"])

    # Scale with level
    hp     = int(base_hp  * (1 + (lvl-1)*0.25))
    damage = int(base_dmg * (1 + (lvl-1)*0.20))
    xp     = random.randint(*t["xp_reward"])  + (lvl-1)*10
    gold   = random.randint(*t["gold_reward"]) + (lvl-1)*5

    # Pick a random loot drop (can be None = no drop)
    loot = random.choice(t["loot_table"])

    return {
        # ── Position / render (for your pygame scene) ──
        "x":        random.uniform(0, W),
        "y":        GROUND_Y,
        "w":        size,
        "h":        size,
        "col":      col,

        # ── Identity ──
        "type":     enemy_type,
        "level":    lvl,
        "flavor":   t["flavor"],

        # ── Combat stats ──
        "hp":       hp,
        "hp_max":   hp,
        "damage":   damage,
        "speed":    speed,
        "abilities":list(t["abilities"]),  # copy so original isn't mutated

        # ── Rewards ──
        "xp_reward":   xp,
        "gold_reward": gold,
        "loot":        loot,

        # ── Status effects tracking ──
        "status": {
            "poisoned":    False,
            "shielded":    False,
            "stunned":     False,
            "poison_turns":0,
        },
    }

# ══════════════════════════════════════════════
#  BATTLE ENGINE
# ══════════════════════════════════════════════

def _print_divider(char="─", width=54):
    print(char * width)

def _show_combatants(player, enemy):
    _print_divider("═")
    print(f"  {player['name'].upper():<22}  VS  {enemy['type'].upper()}")
    _print_divider()

    # HP bars
    def hp_bar(cur, mx, width=20):
        filled = int(width * cur / max(mx,1))
        return "█"*filled + "░"*(width-filled)

    p_bar = hp_bar(player["hp"], player["hp_max"])
    e_bar = hp_bar(enemy["hp"],  enemy["hp_max"])

    print(f"  HP  {p_bar}  {player['hp']}/{player['hp_max']}")
    print(f"  HP  {e_bar}  {enemy['hp']}/{enemy['hp_max']}   [{enemy['type']}]")
    if player.get("stamina") is not None:
        st_bar = hp_bar(player["stamina"], player["stamina_max"])
        print(f"  ST  {st_bar}  {player['stamina']}/{player['stamina_max']}")
    _print_divider("═")

def _show_status(entity, name):
    s = entity.get("status", {})
    flags = []
    if s.get("poisoned"):  flags.append(f"POISONED({s['poison_turns']})")
    if s.get("shielded"):  flags.append("SHIELDED")
    if s.get("stunned"):   flags.append("STUNNED")
    if flags:
        print(f"  ⚠  {name}: {', '.join(flags)}")

def _apply_poison(entity, name):
    """Tick poison damage. Returns damage dealt."""
    s = entity.get("status", {})
    if s.get("poisoned") and s["poison_turns"] > 0:
        dmg = random.randint(2, 5)
        entity["hp"] = max(0, entity["hp"] - dmg)
        s["poison_turns"] -= 1
        if s["poison_turns"] <= 0:
            s["poisoned"] = False
            print(f"  🟢 {name} recovers from poison.")
        else:
            print(f"  ☠  Poison deals {dmg} damage to {name}! "
                  f"({s['poison_turns']} turns left)")
        return dmg
    return 0

def _enemy_turn(player, enemy, player_dodging):
    """
    Resolve the enemy's turn.
    Returns updated (player, enemy) and a result string.
    """
    s = enemy.get("status", {})

    # Stunned — skip turn
    if s.get("stunned"):
        s["stunned"] = False
        print(f"  💤 {enemy['type']} is stunned and loses their turn!")
        return player, enemy

    # Pick ability
    ability_key  = random.choice(enemy["abilities"])
    ability      = ABILITIES[ability_key]
    print(f"\n  ⚔  {enemy['type']} uses {ability['name']}!")
    print(f"     \"{ability['description']}\"")

    # Handle shield_pulse — enemy blocks incoming, no outgoing damage this turn
    if ability_key == "shield_pulse":
        enemy["status"]["shielded"] = True
        print(f"  🛡  {enemy['type']} activates shield — "
              f"takes 50% damage next turn.")
        return player, enemy

    # Calculate raw damage
    raw_dmg = int(enemy["damage"] * ability["damage_mult"])

    # Player dodge attempt
    if player_dodging:
        dodge_roll = random.random()
        if dodge_roll < 0.70:
            print(f"  💨 You dodge the attack!")
            return player, enemy
        else:
            print(f"  ❌ Dodge failed!")

    # Apply damage to player
    if raw_dmg > 0:
        player["hp"] = max(0, player["hp"] - raw_dmg)
        label = "💥 CRITICAL HIT!" if ability_key == "crit_strike" else "🩸"
        print(f"  {label} {enemy['type']} deals {raw_dmg} damage to you.")

    # Poison effect
    if ability["effect"] == "poison":
        player["status"] = player.get("status", {})
        player["status"]["poisoned"]     = True
        player["status"]["poison_turns"] = 3
        print(f"  ☠  You are POISONED for 3 turns!")

    return player, enemy


def _player_turn(player, enemy, action, item_name=None):
    """
    Resolve the player's chosen action.
    Returns (player, enemy, player_is_dodging_this_turn).
    """
    act = PLAYER_ACTIONS.get(action)
    if act is None:
        print("  ⚠  Unknown action. Defaulting to Attack.")
        act = PLAYER_ACTIONS["attack"]

    player_dodging = False

    # Stamina check
    cost = act["stamina_cost"]
    if player.get("stamina") is not None:
        if player["stamina"] < cost:
            print(f"  ⚠  Not enough stamina for {act['name']}! Defaulting to Attack.")
            act  = PLAYER_ACTIONS["attack"]
            cost = act["stamina_cost"]
        player["stamina"] = max(0, player["stamina"] - cost)

    print(f"\n  🎯 You use {act['name']}!")

    # ── Flee ──
    if action == "flee":
        if random.random() < 0.60:
            print("  🏃 You escape successfully!")
            return player, enemy, "fled"
        else:
            print("  ❌ You couldn't escape!")
            return player, enemy, False

    # ── Use item ──
    if action == "use_item":
        inventory = player.get("inventory", [])
        consumables = [i for i in inventory
                       if isinstance(i, dict) and i.get("type") == "consumable"]
        if not consumables:
            print("  ⚠  No consumables in inventory!")
        else:
            # Auto-use first consumable (or pass item_name to target specific)
            item = next((i for i in consumables
                         if item_name and i["name"]==item_name), consumables[0])
            for stat, val in item.get("stats", {}).items():
                if stat == "hp":
                    heal = min(val, player["hp_max"]-player["hp"])
                    player["hp"] += heal
                    print(f"  💊 Used {item['name']} — restored {heal} HP.")
            player["inventory"].remove(item)
        return player, enemy, False

    # ── Dodge (set flag, damage comes later in enemy turn) ──
    if action == "dodge":
        print("  💨 You prepare to dodge!")
        return player, enemy, True   # signal to enemy turn

    # ── Attack / Heavy Strike ──
    base_dmg = player.get("damage", 8)   # fallback if player has no damage stat
    raw_dmg  = int(base_dmg * act["damage_mult"])

    # Enemy shield reduces incoming damage
    if enemy["status"].get("shielded"):
        raw_dmg = raw_dmg // 2
        enemy["status"]["shielded"] = False
        print(f"  🛡  Shield absorbed — reduced to {raw_dmg} damage.")

    if raw_dmg > 0:
        enemy["hp"] = max(0, enemy["hp"] - raw_dmg)
        print(f"  💢 You deal {raw_dmg} damage to {enemy['type']}! "
              f"(HP: {enemy['hp']}/{enemy['hp_max']})")
    else:
        print(f"  〰  Your attack deals no damage.")

    return player, enemy, player_dodging


def _award_victory(player, enemy):
    """Hand out XP, gold, and loot after winning."""
    _print_divider("═")
    print(f"  🏆  VICTORY!  {enemy['type']} defeated!")
    _print_divider()

    xp   = enemy["xp_reward"]
    gold = enemy["gold_reward"]
    loot = enemy["loot"]

    player["stats"]           = player.get("stats", {})
    player["stats"]["enemies_defeated"] = \
        player["stats"].get("enemies_defeated", 0) + 1

    # XP
    player["xp"] = player.get("xp", 0) + xp
    print(f"  ✨ +{xp} XP  (total: {player['xp']})")

    # Gold
    player["gold"] = player.get("gold", 0) + gold
    player["stats"]["gold_earned"] = \
        player["stats"].get("gold_earned", 0) + gold
    print(f"  💰 +{gold} gold  (total: {player['gold']})")

    # Loot drop
    if loot:
        item = {
            "name":        loot,
            "type":        "consumable" if loot == "Stim Pack" else "misc",
            "value":       random.randint(10,60),
            "stats":       {"hp": 40} if loot == "Stim Pack" else {},
            "description": f"Dropped by {enemy['type']}.",
        }
        player["inventory"] = player.get("inventory", [])
        player["inventory"].append(item)
        print(f"  📦 Loot dropped: {loot}")
    else:
        print(f"  📦 No loot this time.")

    _print_divider("═")
    return player


def battle(player, enemy):
    """
    Full turn-based battle loop.

    player dict must have at minimum:
        name, hp, hp_max, damage, gold, xp, inventory, stats
    Optionally: stamina, stamina_max, status

    Returns updated player dict with result key:
        "won" | "lost" | "fled"
    """
    print(f"\n{'═'*54}")
    print(f"  ⚡  ENCOUNTER — {enemy['type'].upper()}")
    print(f"  {enemy['flavor']}")
    print(f"  Level {enemy['level']}  │  "
          f"HP {enemy['hp']}  │  "
          f"Damage {enemy['damage']}  │  "
          f"Abilities: {', '.join(enemy['abilities'])}")
    print(f"{'═'*54}\n")

    # Ensure player has required keys
    player.setdefault("status",    {})
    player.setdefault("inventory", [])
    player.setdefault("gold",      0)
    player.setdefault("xp",        0)
    player.setdefault("damage",    8)
    player.setdefault("stats",     {})

    turn = 0

    while player["hp"] > 0 and enemy["hp"] > 0:
        turn += 1
        print(f"\n  ── Turn {turn} ──────────────────────────────────")
        _show_combatants(player, enemy)
        _show_status(player, player["name"])
        _show_status(enemy,  enemy["type"])

        # ── Poison tick ──
        _apply_poison(player, player["name"])
        _apply_poison(enemy,  enemy["type"])
        if player["hp"] <= 0 or enemy["hp"] <= 0:
            break

        # ── Stamina regen ──
        if player.get("stamina") is not None:
            regen = 6
            player["stamina"] = min(player["stamina_max"],
                                    player["stamina"] + regen)

        # ── Player chooses action ──
        print(f"\n  What do you do?")
        for key, act in PLAYER_ACTIONS.items():
            cost_str = (f"  [stamina: {act['stamina_cost']}]"
                        if act["stamina_cost"] else "")
            print(f"    [{key:<8}]  {act['name']:<16} {cost_str}")
        print(f"    Your stamina: "
              f"{player.get('stamina','—')}/{player.get('stamina_max','—')}")

        action = input("\n  >> ").strip().lower()

        # ── Resolve player action ──
        player, enemy, dodge_flag = _player_turn(player, enemy, action)

        if dodge_flag == "fled":
            player["result"] = "fled"
            return player
        if enemy["hp"] <= 0:
            break

        # ── Resolve enemy action ──
        player_is_dodging = (dodge_flag is True)
        player, enemy = _enemy_turn(player, enemy, player_is_dodging)

    # ── Battle over ──
    print()
    _print_divider("═")
    if enemy["hp"] <= 0:
        player = _award_victory(player, enemy)
        player["result"] = "won"
    else:
        print(f"  💀  YOU HAVE FALLEN.")
        print(f"      {enemy['type']} stands over your body.")
        player["stats"]["times_died"] = \
            player["stats"].get("times_died", 0) + 1
        player["result"] = "lost"
        _print_divider("═")

    return player


# ══════════════════════════════════════════════
#  DEMO
# ══════════════════════════════════════════════
if __name__ == "__main__":

    # ── Build a quick test player ──
    test_player = {
        "name":        "Ghost",
        "hp":          80,
        "hp_max":      80,
        "stamina":     50,
        "stamina_max": 50,
        "damage":      12,
        "gold":        100,
        "xp":          0,
        "inventory":   [
            {"name":"Stim Pack","type":"consumable",
             "value":30,"stats":{"hp":40},
             "description":"Single-use heal."},
        ],
        "status": {},
        "stats":  {},
    }

    # ── Spawn a random enemy ──
    test_enemy = make_enemy(level=2)

    # ── Run the battle ──
    result = battle(test_player, test_enemy)

    print(f"\n  Result : {result['result'].upper()}")
    print(f"  HP left: {result['hp']}/{result['hp_max']}")
    print(f"  Gold   : {result['gold']}")
    print(f"  XP     : {result['xp']}")
    print(f"  Items  : {[i['name'] for i in result['inventory']]}")




<function show_hero at 0x0000018F69442980>
╔══════════════════════════════════════════╗
║       WELCOME TO THE HERO'S JOURNEY      ║
╚══════════════════════════════════════════╝
Please select your hero.

── Existing heroes found ──
  [1] dumbass          Sneak      Lv1  Act 1  Last: 2026-03-21T22:02:17
  [2] pam              Genius     Lv1  Act 1  Last: 2026-03-21T21:51:21
  [3] Sam              Slayer     Lv1  Act 1  Last: 2026-03-21T21:40:57
  [N] Create a new hero
  [D] Delete a hero
[SAVE] Hero 'dumbass' loaded.

════════════════════════════════════════════════════════
  DUMBASS  ·  SNEAK
  Gift: Speed  ·  Magic: Earth  ·  From: Forest
  "raised among ancient trees and hidden paths"
────────────────────────────────────────────────────────
  Level 1  ·  XP 0/100  ·  Act 1
  HP      100/100   Mana    60/60   Stamina 50/50
  Gold    100   Rare Essence  0
────────────────────────────────────────────────────────
  ATTRIBUTES
    strength           █████░░░░░░░░░░░░░░░  5
    intelligenc